#IIDS67692 Computational Techniques for Multi-modal Data
#Lab: Multimodal Large Language Model

In this lab, we will be exploring different fusion techniques in a Multimodal Large Language Model for Health Decision Support

In [1]:
!nvidia-smi

Fri Jul 31 00:52:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:41:00.0 Off |                    0 |
| N/A   25C    P0             57W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### Dependency environment

In [ ]:
# Dependencies are managed in ~/dissertation_2026/.venv before submission.
# Never install or upgrade packages inside an nbconvert job sharing that environment.
import sys
print("Using preconfigured Python environment:", sys.executable)

# Dataset: Complete Official PathVQA Splits

Load the complete official **train**, **validation**, and **test** splits. Training uses only `train`; early stopping uses only `validation`; final utility and PA-SHE safety evaluation use only the held-out `test` split.


In [3]:
from datasets import load_dataset
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Load every official PathVQA split. Keep the test split held out.
pathvqa = load_dataset(
    "parquet",
    data_files={
        "train": "hf://datasets/flaviagiammarino/path-vqa/data/train-*.parquet",
        "validation": "hf://datasets/flaviagiammarino/path-vqa/data/validation-*.parquet",
        "test": "hf://datasets/flaviagiammarino/path-vqa/data/test-*.parquet",
    },
)
pathvqa_train = pathvqa["train"]
pathvqa_validation = pathvqa["validation"]
pathvqa_test = pathvqa["test"]

print({
    "train": len(pathvqa_train),
    "validation": len(pathvqa_validation),
    "test": len(pathvqa_test),
    "total": sum(len(split) for split in pathvqa.values()),
})

idx = 3
print(pathvqa_test[idx].keys())
print("image resolution:", np.array(pathvqa_test[idx]["image"]).shape)
plt.figure(figsize=(5, 5))
plt.imshow(pathvqa_test[idx]["image"])
plt.title(
    f"Q: {pathvqa_test[idx]['question']}\n"
    f"A: {pathvqa_test[idx]['answer']}",
    fontsize=12,
)
plt.axis("off")
plt.show()


{'train': 19654, 'validation': 6259, 'test': 6719, 'total': 32632}
dict_keys(['image', 'question', 'answer'])
image resolution: (324, 492, 4)


#Prepare Dataloader

In [4]:
import torch
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torch.utils.data import DataLoader


class PathVQADataset(Dataset):
    def __init__(self, hf_dataset):
        """
        hf_dataset: HuggingFace dataset (already loaded split)
        """

        self.dataset = hf_dataset

        self.transform = transforms.Compose([
            transforms.Resize((224, 224), interpolation=InterpolationMode.BICUBIC),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]

        # --- Image ---
        # HF dataset already provides PIL image
        raw_image = sample["image"].convert('RGB')
        img = self.transform(raw_image)

        # --- Question & Answer ---
        question = sample["question"]
        answer = sample["answer"]

        return img, question, answer

# Use the complete official splits; do not resplit or subsample.
train_data = pathvqa_train
val_data = pathvqa_validation
test_data = pathvqa_test

train_dataset = PathVQADataset(train_data)
val_dataset = PathVQADataset(val_data)
test_dataset = PathVQADataset(test_data)

print(
    f"Full official splits: train={len(train_dataset)}, "
    f"validation={len(val_dataset)}, test={len(test_dataset)}"
)


img, question, answer = train_dataset[1]
print("image resolution:", img.size())
plt.figure(figsize=(5, 5))
plt.axis("off")
plt.imshow(img.permute(1, 2, 0))
plt.title(f"Q: {question}\nA: {answer}", fontsize=12)
plt.show()


Full official splits: train=19654, validation=6259, test=6719
image resolution: torch.Size([3, 224, 224])


#Model Architecture

Paper: https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf

GPT-2 uses a decoder-only transformer architecture with multiple model sizes; the commonly used GPT-2 Base model contains 12 transformer blocks (layers), a context window of 1024 tokens, a hidden embedding size of 768, and about 117 million parameters, while larger variants scale up to 48 transformer blocks and 1.5 billion parameters.

[1] Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). Language models are unsupervised multitask learners. OpenAI blog, 1(8), 9.

###Cross-Attention Fusion

In [5]:
import math
import torch
import torch.nn as nn
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import ViTModel, BlipTextModel
from peft import get_peft_model

####Cross-Attention Fusion###########
class CrossAttentionFusion(nn.Module):
    def __init__(self, hidden_dim=768, num_heads=8, dropout=0.1):
        super().__init__()

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.Dropout(dropout)
        )

    def forward(self, text_embeds, image_embeds, text_att_mask=None):
        """
        text_embeds:  [B, T, 768]
        image_embeds: [B, N, 768]
        text attends to image
        """

        attended_text, attn_weights = self.cross_attn(
            query=text_embeds,
            key=image_embeds,
            value=image_embeds,
            need_weights=False
        )

        x = self.norm1(text_embeds + attended_text)
        x = self.norm2(x + self.ffn(x))

        return x

W0731 00:53:11.286000 1516496 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W0731 00:53:11.560000 1516496 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


###Gated Cross-Attention Fusion

In [6]:
#Write your code for Gated-Cross Attention Fusion

###MedVQA model (Mulitmodal GPT2)

In [7]:
import math
import torch
import torch.nn as nn
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import ViTModel, BlipTextModel
from peft import get_peft_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class MedVQA(nn.Module):
    def __init__(self, peft_config=None):
        super(MedVQA, self).__init__()

        # visual encoder
        model_name = "google/vit-base-patch16-224-in21k"
        self.visual_encoder = ViTModel.from_pretrained(model_name)

        # Freeze all parameters in visual encoder
        for param in self.visual_encoder.parameters():
            param.requires_grad = False

        # tokenizer
        self.tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
        self.tokenizer.pad_token = self.tokenizer.eos_token  # end of string

        # gpt2 decoder
        gpt = GPT2LMHeadModel.from_pretrained('gpt2')
        self.gpt = get_peft_model(gpt, peft_config)
        # self.gpt.print_trainable_parameters()  # Verify trainable LoRA parameters
        self.fusion = CrossAttentionFusion(
            hidden_dim=768,
            num_heads=4
        )

    def forward(self, image, qa_inputs_ids, qa_att_mask):
        image_embeds = self.visual_encoder(image).last_hidden_state
        # [B, 197, 768]

        text_embeds = self.gpt.get_input_embeddings()(qa_inputs_ids)
        # [B, T, 768]

        fused_embeds = self.fusion(
            text_embeds=text_embeds,
            image_embeds=image_embeds,
            text_att_mask=qa_att_mask
        )
        # [B, T, 768]

        gpt_output = self.gpt(
            inputs_embeds=fused_embeds,
            attention_mask=qa_att_mask
        )
        return gpt_output.logits

# Model training or checkpoint reuse

This notebook reuses its own previously trained **PathVQA PA-SHE** checkpoint by
default:

`checkpoints_joint_se/best_model_ca_joint_se.pth`

If the file is absent, training runs for up to **10 epochs** with validation
early stopping (patience **5**) and saves the best model there. Set
`REUSE_TRAINED_CHECKPOINT=0` to deliberately retrain and overwrite this
notebook's checkpoint. The checkpoint is never shared with the other dataset
or condition-weight experiment.


In [ ]:
#Training Script for Multimodal GPT2 with LoRA
import os
import torch
import argparse
import torch.utils.data
import numpy as np
import random

from torch import nn
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer

import evaluate
from nltk.translate.bleu_score import corpus_bleu
from peft import  TaskType, LoraConfig

import warnings
warnings.filterwarnings('ignore')

REUSE_TRAINED_CHECKPOINT = os.environ.get(
    'REUSE_TRAINED_CHECKPOINT', '1'
).strip().lower() not in {'0', 'false', 'no'}


def load_vqa_checkpoint(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


def adjust_learning_rate(optimizer, shrink_factor):
    print("\nDECAYING learning rate.")
    for param_group in optimizer.param_groups:
        param_group['lr'] = param_group['lr'] * shrink_factor
    print("The new learning rate is %f\n" % (optimizer.param_groups[0]['lr'],))

def train(args, train_dataloader, model, criterion, optimizer, epoch, tokenizer, device):
    model.train()
    total_loss = []

    for i, (images, questions, answers) in enumerate(train_dataloader, 0):
        # prepare prompts
        qa_prompt = [f'Question: {q}\nAnswer: {a}' for q, a in zip(questions, answers)]
        qa_prompt_inputs = tokenizer(qa_prompt, truncation=True, padding="max_length", max_length=int(args.seq_length), return_tensors="pt")

        # get labels
        labels = qa_prompt_inputs['input_ids'].clone()
        labels = labels.to(device)

        # for labels, mask question tokens and padding tokens
        for idx, q in enumerate(questions):
            q_prompt = f"Question: {q}\nAnswer: "
            q_length = len(tokenizer(q_prompt)["input_ids"]) - 1

            labels[idx, :q_length] = -100  # mask question
            eos_mask = (labels[idx] == tokenizer.eos_token_id)  # get all EOS position
            if eos_mask.sum() > 1:  # if more than 1 EOS
                first_eos_pos = eos_mask.nonzero()[0].item()  # get first EOS position
                labels[idx, (first_eos_pos+1):] = -100  # mask paddings, left one EOS

        # get logits and labels
        logits = model(
                image=images.to(device),
                qa_inputs_ids=qa_prompt_inputs['input_ids'].to(device),
                qa_att_mask=qa_prompt_inputs['attention_mask'].to(device)
        )

        # get shifted logits and labels
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        # compute loss
        shift_logits = shift_logits.view(-1, shift_logits.size(-1))
        shift_labels = shift_labels.view(-1)
        loss = criterion(shift_logits, shift_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss.append(loss.item())
        if i%50 == 0:
            print("Training - Epoch: {}/{}, Iteration: {}/{}, Training Loss: {:.6f}".format(epoch, args.epochs, i, len(train_dataloader), np.array(total_loss).mean()))


def validate(args, val_loader, model, criterion, epoch, tokenizer, device):
    total_loss = []
    model.eval()
    with torch.no_grad():
        for i, (images, questions, answers) in enumerate(val_loader, 0):
            # prepare prompts
            qa_prompt = [f'Question: {q}\nAnswer: {a}' for q, a in zip(questions, answers)]
            qa_prompt_inputs = tokenizer(qa_prompt, truncation=True, padding="max_length", max_length=int(args.seq_length), return_tensors="pt")

            # get labels
            labels = qa_prompt_inputs['input_ids'].clone()
            labels = labels.to(device)

            # for labels, mask question tokens and padding tokens
            answer_starts = []
            answer_ends = []
            for idx, q in enumerate(questions):
                q_prompt = f"Question: {q}\nAnswer: "
                q_length = len(tokenizer(q_prompt)["input_ids"]) - 1
                answer_starts.append(q_length+1)

                labels[idx, :q_length] = -100  # mask question
                eos_mask = (labels[idx] == tokenizer.eos_token_id)  # get all EOS position
                if eos_mask.sum() > 1:  # if more than 1 EOS
                    first_eos_pos = eos_mask.nonzero()[0].item()  # get first EOS position
                    labels[idx, (first_eos_pos+1):] = -100  # mask paddings, left one EOS
                    answer_ends.append(first_eos_pos)

            # get logits and labels
            logits = model(
                image=images.to(device),
                qa_inputs_ids=qa_prompt_inputs['input_ids'].to(device),
                qa_att_mask=qa_prompt_inputs['attention_mask'].to(device)
            )

            # get shifted logits and labels
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            # compute loss
            shift_logits = shift_logits.view(-1, shift_logits.size(-1))
            shift_labels = shift_labels.view(-1)
            loss = criterion(shift_logits, shift_labels)
            total_loss.append(loss.item())

    return np.array(total_loss).mean()


def seed_everything(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)


def get_arg():
    parser = argparse.ArgumentParser(description='VisualQuestionAnswerGeneration')
    # Training parameters
    parser.add_argument('--epochs',         type=int,   default=10,   help='number of epochs to train for')
    parser.add_argument('--batch_size',     type=int,   default=64,   help='training and validation batch size')
    parser.add_argument('--workers',        type=int,   default=8,    help='for data-loading')
    parser.add_argument('--random_seed',    type=int,   default=42,   help='random seed')
    parser.add_argument('--seq_length',     type=int,   default=68,   help='sequence length for question and answer')
    parser.add_argument('--dropout', type=float, default=0.1, help='dropout')
    parser.add_argument('--early_stopping_patience', type=int, default=5,
                        help='stop after this many epochs without validation improvement')

    parser.add_argument('--dataset',        default='endo',  help='endo / pit')
    parser.add_argument('--lr',             type=float, default=0.0002,  help='0.0000001, 0.00000005')
    parser.add_argument('--checkpoint_dir', default='checkpoints_joint_se/',
                        help='separate checkpoint path for the PA-SHE experiment')

    args = parser.parse_args([])
    return args


if __name__ == '__main__':

    args = get_arg()
    seed_everything(args.random_seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f'Batch size: {args.batch_size}')
    print(f'Learning rate: {args.lr}')
    print(f'Random seed: {args.random_seed}')
    print(f'Sequence length: {args.seq_length}')
    print(f'Maximum epochs: {args.epochs}')
    print(f'Early-stopping patience: {args.early_stopping_patience}')

    os.makedirs(args.checkpoint_dir, exist_ok=True)
    MODEL_CHECKPOINT_PATH = os.path.join(
        args.checkpoint_dir,
        'best_model_ca_joint_se.pth',
    )
    reuse_checkpoint = (
        REUSE_TRAINED_CHECKPOINT
        and os.path.isfile(MODEL_CHECKPOINT_PATH)
    )
    checkpoint_saved_this_run = False
    start_epoch = 1
    epochs_since_improvement = 0
    best_val_loss = float('inf')

    print('Dataset: full official PathVQA train/validation splits')
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=args.workers > 0,
    )
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=args.workers > 0,
    )

    print(
        'DataLoader configuration:',
        {
            'train_examples': len(train_dataset),
            'validation_examples': len(val_dataset),
            'batch_size': args.batch_size,
            'workers': args.workers,
        },
    )

    # init tokenizer and model
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["c_attn", "c_proj"]
    )

    model = MedVQA(peft_config=lora_config)
    model = model.to(device)

    # for name, param in model.named_parameters():
    #     if param.requires_grad:
    #         print(name)

    pytorch_total_params = sum(p.numel() for p in model.parameters())
    print('model params: ', pytorch_total_params)

    # init optimizer and criterion
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    criterion = nn.CrossEntropyLoss(ignore_index=-100).to(device)

    # Reuse this notebook's validation-selected checkpoint unless retraining
    # is explicitly requested or the checkpoint does not exist.
    if reuse_checkpoint:
        print(
            'Reusing trained checkpoint; skipping epoch training:',
            MODEL_CHECKPOINT_PATH,
        )
        training_epochs = []
    else:
        if REUSE_TRAINED_CHECKPOINT:
            print(
                'No existing checkpoint found; training from scratch:',
                MODEL_CHECKPOINT_PATH,
            )
        else:
            print('Checkpoint reuse disabled; training from scratch.')
        print('Start training.')
        training_epochs = range(start_epoch, args.epochs + 1)

    for epoch in training_epochs:
        if epochs_since_improvement > 0 and epochs_since_improvement % 5 == 0:
            adjust_learning_rate(optimizer, 0.8)

        # train
        train(args, train_dataloader=train_dataloader, model=model, criterion=criterion, optimizer=optimizer,
              epoch=epoch, tokenizer=tokenizer, device=device)
        # validation
        val_loss = validate(args, val_loader=val_dataloader, model=model, criterion=criterion,
                            epoch=epoch, tokenizer=tokenizer, device=device)

        if val_loss < best_val_loss:  # save model with better validation loss
            epochs_since_improvement = 0
            best_val_loss = val_loss
            torch.save(model.state_dict(), MODEL_CHECKPOINT_PATH)
            checkpoint_saved_this_run = True
            model.tokenizer.save_pretrained(args.checkpoint_dir)
            print('Best validation loss, model saved.')
        else:
            epochs_since_improvement += 1
            print("\nEpochs since last improvement: %d\n" % (epochs_since_improvement,))

        if epochs_since_improvement >= args.early_stopping_patience:
            print(
                f'Early stopping at epoch {epoch}: validation loss did not improve '
                f'for {args.early_stopping_patience} consecutive epochs.'
            )
            break

    if not reuse_checkpoint:
        if not checkpoint_saved_this_run:
            raise RuntimeError(
                'Training finished without producing a validation checkpoint.'
            )
        print(f'End training. Best validation loss: {best_val_loss:.6f}')

    if not os.path.isfile(MODEL_CHECKPOINT_PATH):
        raise FileNotFoundError(
            f'Model checkpoint not found: {MODEL_CHECKPOINT_PATH}'
        )
    model.load_state_dict(
        load_vqa_checkpoint(MODEL_CHECKPOINT_PATH, device)
    )
    model.to(device)
    model.eval()
    print('Loaded validation-selected checkpoint:', MODEL_CHECKPOINT_PATH)


# Validation inference: qualitative sanity check

These examples are from validation. The official test split remains untouched
until the dataset-specific PA-SHE clustering has been selected and locked.


In [ ]:
# Validation-only prediction visualisations (test remains untouched)
from tqdm import tqdm
import evaluate

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import ViTModel, BlipTextModel
from peft import get_peft_model
from peft import  TaskType, LoraConfig
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

def greedy_search_single(image, question, model, tokenizer, max_length, device):
    model.eval()
    with torch.no_grad():
        # Prepare prompt and tokenize
        prompt_text = f"Question: {question}\nAnswer:"
        inputs = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False)
        input_ids = inputs['input_ids'].to(device)
        attention_mask = inputs['attention_mask'].to(device)

        # Pad to max_length
        padded_input_ids = torch.zeros((1, max_length), dtype=torch.long, device=device)
        padded_attention_mask = torch.zeros((1, max_length), dtype=torch.long, device=device)
        seq_len = input_ids.size(1)
        padded_input_ids[:, :seq_len] = input_ids
        padded_attention_mask[:, :seq_len] = attention_mask

        valid_length = seq_len
        generated_ids = []

        image = image.unsqueeze(0).to(device)  # Add batch dim

        for _ in range(max_length - seq_len):
            logits = model(
                image=image,
                qa_inputs_ids=padded_input_ids[:, :valid_length],
                qa_att_mask=padded_attention_mask[:, :valid_length]
            )

            last_logits = logits[0, valid_length - 1]  # shape: [vocab_size]
            next_token_id = torch.argmax(F.softmax(last_logits, dim=-1), dim=-1)

            if next_token_id.item() == tokenizer.eos_token_id:
                break

            padded_input_ids[0, valid_length] = next_token_id
            padded_attention_mask[0, valid_length] = 1
            valid_length += 1
            generated_ids.append(next_token_id.item())

        # Decode generated tokens
        answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        return answer


def inference_few_samples(sample_indices = [4, 5, 7]):
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["c_attn", "c_proj"]
    )

    model = MedVQA(peft_config=lora_config)
    save_dir = MODEL_CHECKPOINT_PATH
    # save_dir = f'best_model_ca_lr1.pth'
    model.load_state_dict(load_vqa_checkpoint(save_dir, device))
    model.to(device)
    model.eval()

    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token

    validation_dataset = val_dataset
    print('Validation size (test remains untouched):', len(validation_dataset))

    # Create subplots
    fig, axes = plt.subplots(1, len(sample_indices), figsize=(12, 5))  # 1 row, 3 columns
    for ax, idx in zip(axes, sample_indices):
        img, question, answer_gt = validation_dataset[idx]

        # Run inference
        pred_answer = greedy_search_single(img, question, model, tokenizer, max_length=34, device=device)
        img = img.permute(1, 2, 0)  # Convert from [C, H, W] to [H, W, C] for imshow
        ax.imshow(img)
        ax.set_title(f'Q: {question}\nA: {answer_gt}\nPred: {pred_answer}', fontsize=8)
        ax.axis('off')


inference_few_samples(sample_indices = [2, 80, 7])


# Validation inference: utility sanity check

This is not the final reported test utility. Official-test utility is computed
later from the greedy predictions generated after the clustering lock.


In [ ]:
# Validation-only utility sanity metrics (test remains untouched)
import os
import gc
import numpy as np
import random
import re

import torch
import torch.utils.data
from torch import nn
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.transforms.functional import InterpolationMode
import torch.nn as nn
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import ViTModel, BlipTextModel
from peft import get_peft_model
from peft import  TaskType, LoraConfig

from PIL import Image
from tqdm import tqdm
import evaluate
rouge = evaluate.load("rouge")
import time
import math
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')


def batch_greedy_search(images, questions, model, tokenizer, max_length, device):
    answers = []
    batch_size = len(questions)

    model.eval()
    with torch.no_grad():
        # Prepare the prompts for the entire batch
        prompt_texts = [f"Question: {q}\nAnswer:" for q in questions]

        # Tokenize the prompts with padding to handle varying lengths
        prompt_inputs = tokenizer(
            prompt_texts,
            return_tensors="pt",
            padding='longest',
            add_special_tokens=False
        )

        # Prepare model inputs
        padded_input_ids = torch.zeros((batch_size, max_length), dtype=torch.long, device=device)
        padded_attention_mask = torch.zeros((batch_size, max_length), device=device)

        orig_length = prompt_inputs['input_ids'].size(1)
        padded_input_ids[:, :orig_length] = prompt_inputs['input_ids'].to(device)
        padded_attention_mask[:, :orig_length] = prompt_inputs['attention_mask'].to(device)

        images = images.to(device)

        # Initialize tensors to store generated tokens
        only_answer_ids = torch.empty((batch_size, 0), dtype=torch.long, device=device)

        # Track which sequences have finished generating
        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

        # Record each sample length (number of non-eos tokens)
        valid_lengths = padded_attention_mask.sum(dim=1).long()
        batch_indices = torch.arange(batch_size, device=device)

        for _ in range(max_length - orig_length):
            max_valid_lengths = valid_lengths.max().item()

            logits = model(
                image=images,
                qa_inputs_ids=padded_input_ids[:, :max_valid_lengths],
                qa_att_mask=padded_attention_mask[:, :max_valid_lengths]
            )

            last_valid_logits = logits[batch_indices, valid_lengths - 1, :]
            next_token_ids = torch.argmax(last_valid_logits, dim=-1)

            is_eos = next_token_ids == tokenizer.eos_token_id
            finished = finished | is_eos

            padded_input_ids[batch_indices, valid_lengths] = next_token_ids
            padded_attention_mask[batch_indices, valid_lengths] = 1
            valid_lengths += 1

            only_answer_ids = torch.cat(
                [only_answer_ids, next_token_ids.unsqueeze(1)],
                dim=1
            )

            if finished.all():
                break

        # Decode the generated tokens into strings
        generated_ids_cpu = only_answer_ids.cpu().tolist()  # Move to CPU and convert to list for processing
        for i in range(batch_size):
            # Find the first occurrence of eos_token_id to truncate the answer
            try:
                eos_index = generated_ids_cpu[i].index(tokenizer.eos_token_id)
                answer_ids = generated_ids_cpu[i][:eos_index]
            except ValueError:
                # If eos_token_id is not found, use all generated tokens
                answer_ids = generated_ids_cpu[i]

            # Decode the token IDs to a string, skipping special tokens
            answer = tokenizer.decode(answer_ids, skip_special_tokens=True).strip()
            answers.append(answer)

    return answers

def evaluate_vqa_split(args, data_loader, model, tokenizer, device):
    references = []
    hypotheses = []

    model.eval()
    with torch.no_grad():
        for i, (images, questions, answers) in enumerate(tqdm(data_loader), 0):
            generated_answers = batch_greedy_search(
                images,
                questions,
                model,
                tokenizer,
                max_length=args.seq_length,
                device=device
            )

            references.extend(answers)
            hypotheses.extend(generated_answers)

    return references, hypotheses

def normalize_vqa_metric_text(text):
    text = re.sub(r"[^a-z0-9%.\-\s]", " ", str(text).lower().strip())
    return re.sub(r"\s+", " ", text).strip() or "<empty>"


def get_nlp_mettics(references, hypotheses):
    references = [normalize_vqa_metric_text(text) for text in references]
    hypotheses = [normalize_vqa_metric_text(text) for text in hypotheses]
    bleu = evaluate.load("bleu")
    rouge = evaluate.load("rouge")
    meteor = evaluate.load('meteor')

    # compute HF metrics
    results_bleu = bleu.compute(predictions=hypotheses, references=references)
    results_rouge = rouge.compute(predictions=hypotheses, references=references)
    results_meteor = meteor.compute(predictions=hypotheses, references=references)

    print("HuggingFace Metrics Results:")

    print(f"BLEU-1: {results_bleu['precisions'][0]:.6f}, "
      f"BLEU-2: {results_bleu['precisions'][1]:.6f}, ")

    # print(f"BLEU-4: {results_bleu['bleu']:.6f}")
    print(f"RougeL: {results_rouge['rougeL']:.6f}")
    print(f"Meteor: {results_meteor['meteor']:.6f}")


if __name__ == '__main__':
    # parameters
    random_seed = 42
    seed_everything(random_seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token

    validation_dataset = val_dataset
    utility_eval_batch_size = int(os.environ.get("UTILITY_EVAL_BATCH_SIZE", "32"))
    validation_dataloader = DataLoader(
        validation_dataset, batch_size=utility_eval_batch_size, shuffle=False
    )
    print('Full validation size (test remains untouched):', len(validation_dataset))
    print('Utility evaluation batch size:', utility_eval_batch_size)

    # Reuse the validation-selected model loaded by the training/checkpoint cell.
    if "model" not in globals():
        raise RuntimeError("Run the training/checkpoint cell before utility evaluation.")
    # Training-only state is no longer needed and can retain substantial GPU memory.
    globals().pop("optimizer", None)
    globals().pop("criterion", None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    model.to(device)
    model.eval()

    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token

    validation_references, validation_hypotheses = evaluate_vqa_split(
        args,
        data_loader=validation_dataloader,
        model=model,
        tokenizer=tokenizer,
        device=device,
    )
    get_nlp_mettics(validation_references, validation_hypotheses)


# Perturbation-Aware Semantic Hallucination Entropy (PA-SHE) for PathVQA

The trained VQA model is frozen. PathVQA compares a predeclared 13-candidate
grid: Exact text plus three thresholds each for SBERT, BGE, RoBERTa-NLI,
and DeBERTa-NLI. The candidates are
compared on the official validation split at the fixed failure label
`ROUGE-L < 0.50`. Validation AUROC selects the clustering, validation AUPRC
breaks ties, and the declared candidate order resolves exact ties. The winner
is locked before any PA-SHE test sampling or semantic-feature calculation.

Official test thresholds `0.30` and `0.70` are sensitivity analyses only;
they never re-select clustering.


## Experimental flow

```mermaid
flowchart TB
    A["Official validation split"] --> B["Frozen VQA sampling under four conditions"]
    B --> C1["Exact-text clustering"]
    B --> C2["SBERT and BGE cosine grids<br/>0.70, 0.80, 0.90"]
    B --> C3["RoBERTa/DeBERTa mutual-NLI grids<br/>0.35, 0.50, 0.65"]
    C1 --> D["Validation PA-SHE AUROC<br/>failure = ROUGE-L below 0.50"]
    C2 --> D
    C3 --> D
    D --> E["Lock overall and open-ended winners"]

    F["Official test split"] --> G["Frozen VQA sampling under four conditions"]
    E --> H["Apply locked clustering only"]
    G --> H
    H --> I["Overall and open-ended safety"]
    I --> J["Primary label: ROUGE-L below 0.50"]
    I --> K["Sensitivity only: 0.30 and 0.70"]
```


## 1. Configuration and reproducibility

The complete official validation split is used for clustering selection and
the complete official test split is used once for locked evaluation when
`JSE_MAX_EXAMPLES = None`. Sample and feature caches are tied to the checkpoint,
dataset contents, generation settings, split, and candidate configuration.


In [ ]:
# Install once if needed:
# !pip install -q pandas scipy scikit-learn sentence-transformers transformers rouge-score seaborn

import gc
import os
import hashlib
import json
from pathlib import Path
import math
import random
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TVF
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from rouge_score import rouge_scorer
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import average_precision_score, roc_auc_score
from tqdm.auto import tqdm

JSE_MAX_EXAMPLES = None  # None = every example in each evaluated split
JSE_NUM_SAMPLES = 10
JSE_MAX_NEW_TOKENS = 16
JSE_TEMPERATURE = 1.0
JSE_TOP_P = 0.90
JSE_RANDOM_SEED = 42
JSE_PERTURBATION_VERSION = 2  # invalidates old samples after changing paraphrases/images

# Question-Aligned Semantic Nearest Neighbor Entropy (QA-SNNE).
QA_SNNE_NUM_SAMPLES = int(os.environ.get("QA_SNNE_NUM_SAMPLES", "20"))
QA_SNNE_TEMPERATURE = 1.0
QA_SNNE_TOP_K = 50
QA_SNNE_TOP_P = 0.90
QA_SNNE_BETA = 10.0
QA_SNNE_TAU = 1.0
QA_SNNE_CACHE_SCHEMA_VERSION = 1
QA_SNNE_EMBEDDING_MODEL = "pritamdeka/S-PubMedBert-MS-MARCO"
QA_SNNE_VARIANTS = {
    "Embedding": "qa_snne_embedding",
}
if QA_SNNE_NUM_SAMPLES < 2:
    raise ValueError("QA_SNNE_NUM_SAMPLES must be at least two.")

JSE_LABEL_THRESHOLDS = [0.30, 0.50, 0.70]
JSE_PRIMARY_LABEL_THRESHOLD = 0.50

# Predeclared method/threshold grid. Validation selects; test never does.
JSE_SBERT_THRESHOLDS = [0.70, 0.80, 0.90]
JSE_BGE_THRESHOLDS = [0.70, 0.80, 0.90]
JSE_ROBERTA_NLI_THRESHOLDS = [0.35, 0.50, 0.65]
JSE_DEBERTA_NLI_THRESHOLDS = [0.35, 0.50, 0.65]
LENGTH_ALPHA_GRID = [0.0, 0.5, 1.0]
WEIGHT_TEMPERATURE_GRID = [0.5, 1.0, 2.0, 4.0]
JSE_WEIGHTING_CANDIDATES = [
    (length_alpha, weight_temperature)
    for length_alpha in LENGTH_ALPHA_GRID
    for weight_temperature in WEIGHT_TEMPERATURE_GRID
]
JSE_SELECTION_CANDIDATES = (
    ["Exact text"]
    + [f"SBERT@{threshold:.2f}" for threshold in JSE_SBERT_THRESHOLDS]
    + [f"BGE@{threshold:.2f}" for threshold in JSE_BGE_THRESHOLDS]
    + [
        f"RoBERTa-NLI@{threshold:.2f}"
        for threshold in JSE_ROBERTA_NLI_THRESHOLDS
    ]
    + [
        f"DeBERTa-NLI@{threshold:.2f}"
        for threshold in JSE_DEBERTA_NLI_THRESHOLDS
    ]
)

JSE_SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
JSE_BGE_MODEL = "BAAI/bge-small-en-v1.5"
JSE_ROBERTA_NLI_MODEL = "roberta-large-mnli"
JSE_DEBERTA_NLI_MODEL = "microsoft/deberta-large-mnli"
JSE_MODEL_BATCH_SIZE = 64
JSE_CACHE_SCHEMA_VERSION = 3
JSE_FEATURE_SCHEMA_VERSION = 5

required = [
    "model", "tokenizer", "device", "train_dataset", "val_dataset",
    "test_dataset", "MODEL_CHECKPOINT_PATH",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run the original notebook through model loading/evaluation first. Missing: "
        + ", ".join(missing)
    )

JSE_CHECKPOINT_PATH = Path(MODEL_CHECKPOINT_PATH).resolve()
if not JSE_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {JSE_CHECKPOINT_PATH}")
JSE_CACHE_DIR = (
    JSE_CHECKPOINT_PATH.parent / "joint_se_cache_pathvqa"
)
JSE_CACHE_DIR.mkdir(parents=True, exist_ok=True)


## 2. Perturbations and sampled generation

Four condition families are used: original, weak image, distorted image, and question paraphrase. Paraphrases use type-preserving templates and are accepted only when question type, negation, numbers, and laterality are unchanged. If no template matches, a validated image-context wrapper is used; a rejected rewrite remains unchanged. Pathology images use optical-density H&E stain variation plus bounded scanner gamma, contrast, blur, and noise. No crop or rotation is used, so tissue geometry is preserved.

In [ ]:
def jse_normalize(text):
    return normalize_vqa_metric_text(text)


JSE_QUESTION_ROUTER_VERSION = 1
JSE_CLOSED_QUESTION_PREFIXES = (
    "is " , "are " , "was " , "were " , "do " , "does " ,
    "did " , "can " , "could " , "will " , "would " ,
    "has " , "have " , "had " , "should " , "may " , "might " ,
)


def jse_predict_question_type(question):
    normalized = re.sub(r"\s+", " ", str(question).lower().strip())
    return (
        "Closed"
        if normalized.startswith(JSE_CLOSED_QUESTION_PREFIXES)
        else "Open"
    )


JSE_NEGATION_TERMS = {"no", "not", "without", "absent"}
JSE_LATERALITY_TERMS = {"left", "right", "bilateral"}


def jse_preserved_terms(text, vocabulary):
    tokens = set(re.findall(r"[a-z0-9]+", str(text).lower()))
    return tokens.intersection(vocabulary)


def jse_paraphrase_constraints_hold(original, candidate):
    # Do not let the perturbation change the clinical question or its answer space.
    if not str(candidate).strip():
        return False
    if jse_predict_question_type(original) != jse_predict_question_type(candidate):
        return False
    original_numbers = re.findall(r"\b\d+(?:\.\d+)?\b", str(original))
    candidate_numbers = re.findall(r"\b\d+(?:\.\d+)?\b", str(candidate))
    if original_numbers != candidate_numbers:
        return False
    for protected in (JSE_NEGATION_TERMS, JSE_LATERALITY_TERMS):
        if jse_preserved_terms(original, protected) != jse_preserved_terms(candidate, protected):
            return False
    return jse_normalize(original) != jse_normalize(candidate)


def jse_paraphrase(question):
    q = re.sub(r"\s+", " ", str(question).strip()).rstrip("?")
    if not q:
        return str(question)
    rules = [
        (r"^what does (?:this|the) (?:image|picture) show$", "What is shown in the image?"),
        (r"^what is shown in (?:this|the) (?:image|picture)$", "What does the image show?"),
        (r"^is there (.+)$", r"Does the image show \1?"),
        (r"^does (?:this|the) image show (.+)$", r"Is \1 visible in the image?"),
        (r"^is (.+) present$", r"Does the image show \1?"),
        (r"^are there (.+)$", r"Does the image contain \1?"),
        (r"^can (.+) be seen$", r"Is \1 visible?"),
        (r"^where is (.+) located$", r"What is the location of \1?"),
        (r"^how many (.+) are (?:there|present)$", r"What number of \1 are present?"),
        (r"^what is (?:the )?diagnosis$", "Which diagnosis is most consistent with the image?"),
        (r"^what type of (.+) is (?:this|shown)$", r"Which type of \1 is shown?"),
        (r"^what is present$", "What finding is present?"),
    ]
    for pattern, replacement in rules:
        match = re.fullmatch(pattern, q, flags=re.IGNORECASE)
        if match:
            candidate = match.expand(replacement).strip()
            if jse_paraphrase_constraints_hold(q, candidate):
                return candidate
    first_word = q.split(maxsplit=1)[0].lower()
    open_body = q[0].lower() + q[1:] if first_word in {"what", "where", "when", "why", "who", "which", "how"} else q
    fallback = (
        f"{q} according to the image?"
        if jse_predict_question_type(q) == "Closed"
        else f"Based on the image, {open_body}?"
    )
    if jse_paraphrase_constraints_hold(q, fallback):
        return fallback
    # Reject any unsafe rewrite instead of silently changing semantics.
    return f"{q}?"


JSE_HE_STAIN_BASIS = torch.tensor(
    [[0.650, 0.704], [0.072, 0.990], [0.268, 0.105]], dtype=torch.float32
)
JSE_HE_STAIN_PINV = torch.linalg.pinv(JSE_HE_STAIN_BASIS)


def jse_rng(seed):
    return random.Random(int(seed)) if seed is not None else random


def jse_he_stain_shift(image, haematoxylin_scale, eosin_scale):
    x = image.detach().cpu().float().clamp(0, 1)
    if x.ndim != 3 or x.shape[0] < 3:
        return x
    rgb = x[:3].clamp(min=1.0 / 255.0)
    optical_density = -torch.log(rgb).reshape(3, -1)
    concentrations = (JSE_HE_STAIN_PINV @ optical_density).clamp_min(0)
    base_od = JSE_HE_STAIN_BASIS @ concentrations
    residual_od = optical_density - base_od
    scales = torch.tensor(
        [haematoxylin_scale, eosin_scale], dtype=concentrations.dtype
    ).unsqueeze(1)
    shifted_od = JSE_HE_STAIN_BASIS @ (concentrations * scales) + residual_od
    shifted = torch.exp(-shifted_od).reshape_as(rgb)
    result = x.clone()
    result[:3] = shifted
    return result.clamp(0, 1)


def jse_noise_like(image, standard_deviation, seed):
    generator = torch.Generator(device="cpu")
    generator.manual_seed(int(seed))
    return torch.randn(image.shape, generator=generator, dtype=image.dtype) * standard_deviation


def jse_weak_image(image, seed=None):
    rng = jse_rng(seed)
    x = jse_he_stain_shift(image, rng.uniform(0.95, 1.05), rng.uniform(0.95, 1.05))
    x = TVF.adjust_gamma(x, rng.uniform(0.95, 1.05))
    x = TVF.adjust_contrast(x, rng.uniform(0.97, 1.03))
    sigma = rng.uniform(0.10, 0.35)
    return TVF.gaussian_blur(x, [3, 3], [sigma, sigma]).clamp(0, 1)


def jse_distorted_image(image, seed=None):
    rng = jse_rng(seed)
    x = jse_he_stain_shift(image, rng.uniform(0.82, 1.18), rng.uniform(0.82, 1.18))
    x = TVF.adjust_gamma(x, rng.uniform(0.85, 1.15))
    x = TVF.adjust_contrast(x, rng.uniform(0.90, 1.10))
    sigma = rng.uniform(0.45, 1.10)
    x = TVF.gaussian_blur(x, [5, 5], [sigma, sigma])
    noise_seed = int(seed) + 1 if seed is not None else random.randrange(2**31)
    x = x + jse_noise_like(x, rng.uniform(0.01, 0.035), noise_seed)
    return x.clamp(0, 1)


def jse_top_p_filter(logits, top_p):
    if top_p >= 1.0:
        return logits
    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
    cumulative = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
    remove = cumulative > top_p
    remove[..., 1:] = remove[..., :-1].clone()
    remove[..., 0] = False
    sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
    filtered = torch.full_like(logits, float("-inf"))
    return filtered.scatter(-1, sorted_indices, sorted_logits)


@torch.inference_mode()
def jse_generate_one(image, question, do_sample=True):
    prompt = f"Question: {question}\nAnswer:"
    encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    input_ids = encoded["input_ids"].to(device)
    attention = encoded["attention_mask"].to(device)
    image_batch = image.unsqueeze(0).to(device)
    generated, logps = [], []

    for _ in range(JSE_MAX_NEW_TOKENS):
        logits = model(
            image=image_batch,
            qa_inputs_ids=input_ids,
            qa_att_mask=attention,
        )[0, -1]
        scaled = logits / JSE_TEMPERATURE
        log_probs = torch.log_softmax(scaled, dim=-1)
        if do_sample:
            filtered = jse_top_p_filter(scaled, JSE_TOP_P)
            next_id = torch.distributions.Categorical(logits=filtered).sample()
        else:
            next_id = torch.argmax(scaled)
        token_id = int(next_id.item())
        if token_id == tokenizer.eos_token_id:
            break
        generated.append(token_id)
        logps.append(float(log_probs[token_id].item()))
        input_ids = torch.cat([input_ids, next_id.view(1, 1)], dim=1)
        attention = torch.cat(
            [attention, torch.ones((1, 1), dtype=attention.dtype, device=device)],
            dim=1,
        )

    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return {
        "answer": answer,
        "sequence_logprob": float(np.sum(logps)) if logps else -50.0,
    }


@torch.inference_mode()
def qa_snne_generate_samples(image, question, num_samples):
    """Generate QA-SNNE samples from the original input in one batch."""
    prompt = f"Question: {question}\nAnswer:"
    encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    input_ids = encoded["input_ids"].to(device).repeat(num_samples, 1)
    attention = encoded["attention_mask"].to(device).repeat(num_samples, 1)
    image_batch = image.unsqueeze(0).to(device).repeat(num_samples, 1, 1, 1)
    generated_ids = [[] for _ in range(num_samples)]
    finished = torch.zeros(num_samples, dtype=torch.bool, device=device)
    for _ in range(JSE_MAX_NEW_TOKENS):
        next_logits = model(
            image=image_batch, qa_inputs_ids=input_ids, qa_att_mask=attention
        )[:, -1, :] / QA_SNNE_TEMPERATURE
        top_k = min(QA_SNNE_TOP_K, next_logits.shape[-1])
        if top_k > 0:
            kth = torch.topk(next_logits, top_k, dim=-1).values[:, -1:]
            next_logits = next_logits.masked_fill(
                next_logits < kth, float("-inf")
            )
        next_logits = jse_top_p_filter(next_logits, QA_SNNE_TOP_P)
        next_ids = torch.distributions.Categorical(logits=next_logits).sample()
        next_ids = torch.where(
            finished, torch.full_like(next_ids, tokenizer.eos_token_id), next_ids
        )
        for sample_index, token_id in enumerate(next_ids.detach().cpu().tolist()):
            if not finished[sample_index] and token_id != tokenizer.eos_token_id:
                generated_ids[sample_index].append(token_id)
        finished = finished | (next_ids == tokenizer.eos_token_id)
        input_ids = torch.cat([input_ids, next_ids[:, None]], dim=1)
        attention = torch.cat([
            attention,
            torch.ones((num_samples, 1), dtype=attention.dtype, device=device),
        ], dim=1)
        if bool(finished.all()):
            break
    return [
        tokenizer.decode(token_ids, skip_special_tokens=True).strip()
        for token_ids in generated_ids
    ]


def jse_collect_example(
    dataset,
    dataset_index,
    split_name,
):
    image, question, reference = dataset[dataset_index]
    paraphrase = jse_paraphrase(question)
    split_offset = 0 if str(split_name).lower().startswith("val") else 10_000_000
    perturbation_seed = JSE_RANDOM_SEED + split_offset + int(dataset_index) * 1000
    condition_inputs = {
        "original": [(image.clone(), question) for _ in range(JSE_NUM_SAMPLES)],
        "weak": [
            (jse_weak_image(image, perturbation_seed + 100 + sample_index), question)
            for sample_index in range(JSE_NUM_SAMPLES)
        ],
        "distorted": [
            (jse_distorted_image(image, perturbation_seed + 200 + sample_index), question)
            for sample_index in range(JSE_NUM_SAMPLES)
        ],
        "paraphrase": [(image.clone(), paraphrase) for _ in range(JSE_NUM_SAMPLES)],
    }
    records = []
    for condition, inputs in condition_inputs.items():
        for condition_image, condition_question in inputs:
            record = jse_generate_one(condition_image, condition_question, do_sample=True)
            record["condition"] = condition
            records.append(record)
    greedy = jse_generate_one(
        image,
        question,
        do_sample=False,
    )["answer"]
    return {
        "cache_schema_version": JSE_CACHE_SCHEMA_VERSION,
        "split": str(split_name),
        "dataset_index": int(dataset_index),
        "question": question,
        "paraphrase": paraphrase,
        "paraphrase_changed": jse_normalize(question) != jse_normalize(paraphrase),
        "reference": str(reference),
        "greedy": greedy,
        "records": records,
    }


## 3. Dataset-specific clustering backends

PathVQA validation compares Exact text with SBERT/BGE cosine thresholds
`0.70`, `0.80`, and `0.90`, plus bidirectional RoBERTa/DeBERTa-NLI
thresholds `0.35`, `0.50`, and `0.65`. Each method's pairwise score matrix is
calculated once per example and reused across its thresholds. Official test
constructs only the validation-locked overall and open-ended configurations.


In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

jse_sbert = SentenceTransformer(JSE_SBERT_MODEL, device=str(device))
jse_bge = SentenceTransformer(JSE_BGE_MODEL, device=str(device))


def jse_load_nli(model_name):
    tokenizer_nli = AutoTokenizer.from_pretrained(model_name)
    model_nli = AutoModelForSequenceClassification.from_pretrained(
        model_name
    ).to(device).eval()
    entailment_id = next(
        (
            int(index)
            for index, label in model_nli.config.id2label.items()
            if "entail" in str(label).lower()
        ),
        2,
    )
    return tokenizer_nli, model_nli, entailment_id


jse_roberta_tok, jse_roberta, jse_roberta_entail = jse_load_nli(
    JSE_ROBERTA_NLI_MODEL
)
jse_deberta_tok, jse_deberta, jse_deberta_entail = jse_load_nli(
    JSE_DEBERTA_NLI_MODEL
)


def jse_unique_answers(answers):
    normalized = [jse_normalize(answer) for answer in answers]
    unique = list(dict.fromkeys(normalized))
    return normalized, unique


def jse_exact_clusters(answers):
    normalized, unique = jse_unique_answers(answers)
    mapping = {answer: index for index, answer in enumerate(unique)}
    return [mapping[answer] for answer in normalized]


def jse_embedding_cache(answers, encoder):
    normalized, unique = jse_unique_answers(answers)
    embeddings = encoder.encode(
        unique,
        normalize_embeddings=True,
        batch_size=JSE_MODEL_BATCH_SIZE,
    )
    matrix = np.asarray(embeddings) @ np.asarray(embeddings).T
    return normalized, unique, matrix


@torch.inference_mode()
def jse_nli_cache(answers, tokenizer_nli, model_nli, entailment_id):
    normalized, unique = jse_unique_answers(answers)
    scores = np.eye(len(unique), dtype=float)
    pairs = [
        (i, j)
        for i in range(len(unique))
        for j in range(len(unique))
        if i != j
    ]
    for start in range(0, len(pairs), JSE_MODEL_BATCH_SIZE):
        batch = pairs[start:start + JSE_MODEL_BATCH_SIZE]
        encoded = tokenizer_nli(
            [unique[i] for i, _ in batch],
            [unique[j] for _, j in batch],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        ).to(device)
        probabilities = torch.softmax(
            model_nli(**encoded).logits,
            dim=-1,
        )[:, entailment_id]
        for (i, j), value in zip(batch, probabilities.cpu().tolist()):
            scores[i, j] = value
    return normalized, unique, scores


def jse_clusters_from_similarity(cache, threshold, bidirectional=False):
    normalized, unique, matrix = cache
    representatives = []
    unique_cluster_ids = []
    for i in range(len(unique)):
        assigned = None
        for cluster_id, representative in enumerate(representatives):
            forward = matrix[i, representative] >= threshold
            backward = matrix[representative, i] >= threshold
            if forward and (backward if bidirectional else True):
                assigned = cluster_id
                break
        if assigned is None:
            assigned = len(representatives)
            representatives.append(i)
        unique_cluster_ids.append(assigned)
    mapping = dict(zip(unique, unique_cluster_ids))
    return [mapping[item] for item in normalized]

_qa_snne_embedding_encoder = None


def qa_snne_get_embedding_encoder():
    global _qa_snne_embedding_encoder
    if _qa_snne_embedding_encoder is None:
        _qa_snne_embedding_encoder = SentenceTransformer(
            QA_SNNE_EMBEDDING_MODEL, device=str(device)
        )
    return _qa_snne_embedding_encoder


def qa_snne_embedding_alignment(question, answers):
    encoder = qa_snne_get_embedding_encoder()
    embeddings = np.asarray(encoder.encode(
        [str(question)] + [str(answer) for answer in answers],
        normalize_embeddings=True,
        batch_size=JSE_MODEL_BATCH_SIZE,
    ))
    return embeddings[1:] @ embeddings[0]



## 4. Risk definitions

For condition \(k\), sampled sequence log-probabilities are length- and temperature-calibrated, normalised within that condition, and accumulated by semantic cluster:

\[
p_k(c)=\frac{\sum_{s\in k,\ z(s)=c}\exp(\ell_s/(L_s^\alpha T))}
{\sum_{s\in k}\exp(\ell_s/(L_s^\alpha T))}.
\]

With \(K\) available conditions, PA-SHE uses \(\bar p(c)=K^{-1}\sum_k p_k(c)\) and \(H_{\mathrm{joint}}=-\sum_c\bar p(c)\log\bar p(c)\). SE uses only \(p_{\mathrm{original}}\). VASE is the Jensen–Shannon divergence between weak and distorted condition distributions. SNNE uses pairwise ROUGE-L among 20 original-input samples; QA-SNNE reweights those similarities using question–answer embedding alignment.

In [14]:
JSE_CONDITIONS = ["original", "weak", "distorted", "paraphrase"]


def jse_entropy(probabilities):
    p = np.asarray(probabilities, dtype=float)
    p = p[p > 0]
    return float(-(p * np.log(p + 1e-12)).sum())


def jse_record_sequence_length(record):
    if "_sequence_length" not in record:
        record["_sequence_length"] = max(1, len(tokenizer.encode(
            str(record.get("answer", "")), add_special_tokens=False
        )))
    return int(record["_sequence_length"])


def jse_distribution(
    records, cluster_ids, condition, cluster_count,
    length_alpha, weight_temperature,
):
    indices = [i for i, r in enumerate(records) if r["condition"] == condition]
    p = np.zeros(cluster_count, dtype=float)
    if not indices:
        return p
    if length_alpha < 0 or weight_temperature <= 0:
        raise ValueError("Invalid sequence-probability calibration.")
    logps = np.asarray([records[i]["sequence_logprob"] for i in indices])
    lengths = np.asarray([
        jse_record_sequence_length(records[i]) for i in indices
    ], dtype=float)
    calibrated = logps / np.power(lengths, float(length_alpha))
    calibrated = calibrated / float(weight_temperature)
    weights = np.exp(calibrated - calibrated.max())
    weights /= max(weights.sum(), 1e-12)
    for i, weight in zip(indices, weights):
        p[int(cluster_ids[i])] += float(weight)
    return p / max(p.sum(), 1e-12)


def jse_signals(
    example, all_cluster_ids, length_alpha, weight_temperature,
):
    records = example["records"]
    record_ids = all_cluster_ids
    cluster_count = max(all_cluster_ids) + 1
    distributions = {
        condition: jse_distribution(
            records, record_ids, condition, cluster_count,
            length_alpha, weight_temperature,
        )
        for condition in JSE_CONDITIONS
    }
    available = [p for p in distributions.values() if p.sum() > 0]
    joint = np.mean(available, axis=0)
    p_original = distributions["original"]
    p_weak, p_distorted = distributions["weak"], distributions["distorted"]
    vase = float(jensenshannon(
        p_weak + 1e-12, p_distorted + 1e-12, base=2.0
    ) ** 2)
    return {
        "vase": vase,
        "semantic_entropy": jse_entropy(p_original),
        "joint_se": jse_entropy(joint),
        "cluster_count": int(cluster_count),
    }


jse_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def jse_rouge_l(reference, prediction):
    reference = normalize_vqa_metric_text(reference)
    prediction = normalize_vqa_metric_text(prediction)
    return float(jse_rouge.score(reference, prediction)["rougeL"].fmeasure)

def qa_snne_rouge_similarity_matrix(answers):
    answers = [jse_normalize(answer) for answer in answers]
    n = len(answers)
    matrix = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(i + 1, n):
            forward = jse_rouge_l(answers[i], answers[j])
            backward = jse_rouge_l(answers[j], answers[i])
            matrix[i, j] = matrix[j, i] = 0.5 * (forward + backward)
    return matrix


def qa_snne_score(similarity_matrix, alignment_scores=None):
    """Equations (1)-(4) of Carlini et al.; higher means less certain."""
    similarity = np.asarray(similarity_matrix, dtype=np.float64)
    n = similarity.shape[0]
    if similarity.shape != (n, n) or n < 2:
        raise ValueError("SNNE requires a square matrix with at least two answers.")
    if alignment_scores is not None:
        alignment = np.asarray(alignment_scores, dtype=np.float64)
        if alignment.shape != (n,):
            raise ValueError("QA-SNNE alignment scores must match sampled answers.")
        shifted = QA_SNNE_BETA * alignment
        shifted -= shifted.max()
        relevance = np.exp(shifted)
        relevance /= max(relevance.sum(), 1e-12)
        similarity = np.diag(relevance) @ similarity @ np.diag(relevance)

    row_log_sums = []
    for i in range(n):
        values = np.delete(similarity[i], i) / QA_SNNE_TAU
        maximum = float(values.max())
        row_log_sums.append(
            maximum + np.log(np.exp(values - maximum).sum() + 1e-12)
        )
    return float(-np.mean(row_log_sums))


def qa_snne_signals(example, qa_sample_example):
    answers = [str(answer) for answer in qa_sample_example["answers"]]
    if len(answers) != QA_SNNE_NUM_SAMPLES:
        raise ValueError("QA-SNNE sample count does not match configuration.")
    question = str(example["question"])
    similarity = qa_snne_rouge_similarity_matrix(answers)
    embedding_alignment = qa_snne_embedding_alignment(question, answers)
    return {
        "snne": qa_snne_score(similarity),
        "qa_snne_embedding": qa_snne_score(similarity, embedding_alignment),
        "qa_snne_embedding_alignment_mean": float(np.mean(embedding_alignment)),
    }

## 5. Validation sampling and cache

Only validation examples are sampled in this cell. The official test split is
not sampled until section 7 has selected and locked a clustering rule.


In [ ]:
JSE_CONDITIONS = ["original", "weak", "distorted", "paraphrase"]
jse_datasets = {
    "validation": val_dataset,
    "test": test_dataset,
}


def jse_dataset_signature(dataset, evaluation_size):
    digest = hashlib.sha256()
    for dataset_index in range(evaluation_size):
        raw = dataset.dataset[dataset_index]
        digest.update(str(dataset_index).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(raw["question"]).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(raw["answer"]).encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()


def jse_collect_split(split_name, dataset):
    evaluation_size = (
        len(dataset)
        if JSE_MAX_EXAMPLES is None
        else min(int(JSE_MAX_EXAMPLES), len(dataset))
    )
    checkpoint_stat = JSE_CHECKPOINT_PATH.stat()
    cache_configuration = {
        "cache_schema_version": JSE_CACHE_SCHEMA_VERSION,
        "split": split_name,
        "evaluation_size": evaluation_size,
        "dataset_signature": jse_dataset_signature(dataset, evaluation_size),
        "checkpoint_path": str(JSE_CHECKPOINT_PATH),
        "checkpoint_size": int(checkpoint_stat.st_size),
        "checkpoint_mtime_ns": int(checkpoint_stat.st_mtime_ns),
        "samples_per_condition": JSE_NUM_SAMPLES,
        "maximum_new_tokens": JSE_MAX_NEW_TOKENS,
        "temperature": JSE_TEMPERATURE,
        "top_p": JSE_TOP_P,
        "seed": JSE_RANDOM_SEED,
        "perturbation_version": JSE_PERTURBATION_VERSION,
        "conditions": JSE_CONDITIONS,
    }
    cache_hash = hashlib.sha256(
        json.dumps(cache_configuration, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    cache_path = JSE_CACHE_DIR / f"{split_name}_samples_{cache_hash}.jsonl"

    cached_examples = {}
    if cache_path.exists():
        with cache_path.open("r", encoding="utf-8") as cache_file:
            for line_number, line in enumerate(cache_file, start=1):
                if not line.strip():
                    continue
                try:
                    example = json.loads(line)
                except json.JSONDecodeError:
                    print(f"Ignoring incomplete cache line {line_number}: {cache_path}")
                    continue
                dataset_index = int(example.get("dataset_index", -1))
                if example.get("cache_schema_version") != JSE_CACHE_SCHEMA_VERSION:
                    raise ValueError("PA-SHE sample-cache schema mismatch.")
                if example.get("split") != split_name:
                    raise ValueError("PA-SHE sample-cache split mismatch.")
                if not 0 <= dataset_index < evaluation_size:
                    raise ValueError("Cached dataset index is outside this run.")
                condition_counts = {
                    condition: sum(
                        record.get("condition") == condition
                        for record in example.get("records", [])
                    )
                    for condition in JSE_CONDITIONS
                }
                if any(
                    count != JSE_NUM_SAMPLES
                    for count in condition_counts.values()
                ):
                    raise ValueError("Cached condition/sample counts do not match.")
                if dataset_index in cached_examples:
                    raise ValueError("Duplicate dataset index in PA-SHE cache.")
                cached_examples[dataset_index] = example

    pending_indices = [
        index for index in range(evaluation_size)
        if index not in cached_examples
    ]
    print({
        "split": split_name,
        "sample_cache": str(cache_path),
        "cached_examples": len(cached_examples),
        "pending_examples": len(pending_indices),
    })

    split_seed_offset = 0 if split_name == "validation" else 10_000_000
    with cache_path.open("a", encoding="utf-8") as cache_file:
        for dataset_index in tqdm(
            pending_indices,
            desc=f"Frozen-VQA sampling: {split_name}",
        ):
            example_seed = JSE_RANDOM_SEED + split_seed_offset + dataset_index * 1009
            random.seed(example_seed)
            np.random.seed(example_seed)
            torch.manual_seed(example_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(example_seed)
            example = jse_collect_example(
                dataset=dataset,
                dataset_index=dataset_index,
                split_name=split_name,
            )
            cache_file.write(json.dumps(example, ensure_ascii=False) + "\n")
            cache_file.flush()
            cached_examples[dataset_index] = example

    return (
        [cached_examples[index] for index in range(evaluation_size)],
        cache_path,
    )


# Selection starts with validation only. Test sampling occurs in section 7,
# after JSE_LOCKED_CLUSTERING has been assigned.
def qa_snne_collect_split(split_name, dataset):
    evaluation_size = (
        len(dataset)
        if JSE_MAX_EXAMPLES is None
        else min(int(JSE_MAX_EXAMPLES), len(dataset))
    )
    checkpoint_stat = JSE_CHECKPOINT_PATH.stat()
    configuration = {
        "cache_schema_version": QA_SNNE_CACHE_SCHEMA_VERSION,
        "split": split_name,
        "evaluation_size": evaluation_size,
        "dataset_signature": jse_dataset_signature(dataset, evaluation_size),
        "checkpoint_path": str(JSE_CHECKPOINT_PATH),
        "checkpoint_size": int(checkpoint_stat.st_size),
        "checkpoint_mtime_ns": int(checkpoint_stat.st_mtime_ns),
        "num_samples": QA_SNNE_NUM_SAMPLES,
        "maximum_new_tokens": JSE_MAX_NEW_TOKENS,
        "temperature": QA_SNNE_TEMPERATURE,
        "top_k": QA_SNNE_TOP_K,
        "top_p": QA_SNNE_TOP_P,
        "seed": JSE_RANDOM_SEED,
        "input_condition": "original image and original question",
    }
    cache_hash = hashlib.sha256(
        json.dumps(configuration, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    cache_path = JSE_CACHE_DIR / f"{split_name}_qa_snne_samples_{cache_hash}.jsonl"
    cached = {}
    if cache_path.exists():
        with cache_path.open("r", encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, start=1):
                if not line.strip():
                    continue
                try:
                    record = json.loads(line)
                except json.JSONDecodeError:
                    print(f"Ignoring incomplete QA-SNNE cache line {line_number}")
                    continue
                index = int(record.get("dataset_index", -1))
                if record.get("cache_schema_version") != QA_SNNE_CACHE_SCHEMA_VERSION:
                    raise ValueError("QA-SNNE sample-cache schema mismatch.")
                if record.get("split") != split_name or not 0 <= index < evaluation_size:
                    raise ValueError("QA-SNNE sample-cache split/index mismatch.")
                if len(record.get("answers", [])) != QA_SNNE_NUM_SAMPLES:
                    raise ValueError("QA-SNNE cached sample count mismatch.")
                if index in cached:
                    raise ValueError("Duplicate index in QA-SNNE sample cache.")
                cached[index] = record

    pending = [index for index in range(evaluation_size) if index not in cached]
    print({
        "split": split_name,
        "qa_snne_sample_cache": str(cache_path),
        "cached_examples": len(cached),
        "pending_examples": len(pending),
        "samples_per_example": QA_SNNE_NUM_SAMPLES,
    })
    split_offset = 30_000_000 if split_name == "validation" else 40_000_000
    with cache_path.open("a", encoding="utf-8") as handle:
        for index in tqdm(pending, desc=f"QA-SNNE sampling: {split_name}"):
            seed = JSE_RANDOM_SEED + split_offset + index * 1013
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
            image, question, _ = dataset[index]
            record = {
                "cache_schema_version": QA_SNNE_CACHE_SCHEMA_VERSION,
                "split": split_name,
                "dataset_index": int(index),
                "question": str(question),
                "answers": qa_snne_generate_samples(
                    image, question, QA_SNNE_NUM_SAMPLES
                ),
            }
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            handle.flush()
            cached[index] = record
    return [cached[index] for index in range(evaluation_size)], cache_path

jse_validation_examples, jse_validation_sample_cache_path = jse_collect_split(
    "validation",
    jse_datasets["validation"],
)
jse_examples_by_split = {"validation": jse_validation_examples}
jse_sample_cache_paths = {"validation": jse_validation_sample_cache_path}
qa_snne_validation_examples, qa_snne_validation_sample_cache_path = (
    qa_snne_collect_split("validation", jse_datasets["validation"])
)
qa_snne_examples_by_split = {"validation": qa_snne_validation_examples}
qa_snne_sample_cache_paths = {
    "validation": qa_snne_validation_sample_cache_path
}

print({
    "validation_examples": len(jse_validation_examples),
    "test_sampled_before_selection": False,
})
display(pd.DataFrame([{
    "split": example["split"],
    "index": example["dataset_index"],
    "question": example["question"],
    "reference": example["reference"],
    "greedy": example["greedy"],
} for example in jse_validation_examples[:5]]))


## 6. Validation candidate features

Build validation features for the predeclared 13-candidate grid: Exact text
and three thresholds for each of SBERT, BGE, RoBERTa-NLI, and DeBERTa-NLI.
The same cache also stores VASE, SNNE, and all QA-SNNE variants.
No test feature exists yet.


In [ ]:
def jse_clusters_for_configurations(answers, clustering_configurations):
    clustering_configurations = list(clustering_configurations)
    unsupported = set(clustering_configurations) - set(JSE_SELECTION_CANDIDATES)
    if unsupported:
        raise ValueError(
            "Unsupported PathVQA clustering configuration(s): "
            + ", ".join(sorted(unsupported))
        )

    requested = set(clustering_configurations)
    configurations = []
    if "Exact text" in requested:
        configurations.append(("Exact text", jse_exact_clusters(answers)))

    embedding_specs = [
        ("SBERT", JSE_SBERT_THRESHOLDS, jse_sbert),
        ("BGE", JSE_BGE_THRESHOLDS, jse_bge),
    ]
    for method_name, thresholds, encoder in embedding_specs:
        requested_thresholds = [
            threshold for threshold in thresholds
            if f"{method_name}@{threshold:.2f}" in requested
        ]
        if requested_thresholds:
            score_cache = jse_embedding_cache(answers, encoder)
            for threshold in requested_thresholds:
                configuration_name = f"{method_name}@{threshold:.2f}"
                configurations.append((
                    configuration_name,
                    jse_clusters_from_similarity(
                        score_cache, threshold, bidirectional=False
                    ),
                ))

    nli_specs = [
        (
            "RoBERTa-NLI", JSE_ROBERTA_NLI_THRESHOLDS,
            jse_roberta_tok, jse_roberta, jse_roberta_entail,
        ),
        (
            "DeBERTa-NLI", JSE_DEBERTA_NLI_THRESHOLDS,
            jse_deberta_tok, jse_deberta, jse_deberta_entail,
        ),
    ]
    for method_name, thresholds, tok, mdl, entail_id in nli_specs:
        requested_thresholds = [
            threshold for threshold in thresholds
            if f"{method_name}@{threshold:.2f}" in requested
        ]
        if requested_thresholds:
            score_cache = jse_nli_cache(answers, tok, mdl, entail_id)
            for threshold in requested_thresholds:
                configuration_name = f"{method_name}@{threshold:.2f}"
                configurations.append((
                    configuration_name,
                    jse_clusters_from_similarity(
                        score_cache, threshold, bidirectional=True
                    ),
                ))
    if {name for name, _ in configurations} != set(clustering_configurations):
        raise ValueError("Failed to construct every requested clustering.")
    return configurations


def jse_build_feature_frame(
    split_name,
    examples,
    sample_cache_path,
    clustering_configurations,
    weighting_configurations,
):
    clustering_configurations = list(clustering_configurations)
    weighting_configurations = [tuple(item) for item in weighting_configurations]
    feature_configuration = {
        "feature_schema_version": JSE_FEATURE_SCHEMA_VERSION,
        "metric_normalization_version": 1,
        "question_router_version": JSE_QUESTION_ROUTER_VERSION,
        "split": split_name,
        "sample_cache": sample_cache_path.name,
        "clustering_configurations": clustering_configurations,
        "weighting_configurations": weighting_configurations,
        "sbert_model": JSE_SBERT_MODEL,
        "sbert_thresholds": JSE_SBERT_THRESHOLDS,
        "bge_model": JSE_BGE_MODEL,
        "bge_thresholds": JSE_BGE_THRESHOLDS,
        "roberta_nli_model": JSE_ROBERTA_NLI_MODEL,
        "roberta_nli_thresholds": JSE_ROBERTA_NLI_THRESHOLDS,
        "deberta_nli_model": JSE_DEBERTA_NLI_MODEL,
        "deberta_nli_thresholds": JSE_DEBERTA_NLI_THRESHOLDS,
        "qa_snne_sample_cache": qa_snne_sample_cache_paths[split_name].name,
        "qa_snne_num_samples": QA_SNNE_NUM_SAMPLES,
        "qa_snne_beta": QA_SNNE_BETA,
        "qa_snne_tau": QA_SNNE_TAU,
        "qa_snne_embedding_model": QA_SNNE_EMBEDDING_MODEL,
    }
    feature_hash = hashlib.sha256(
        json.dumps(feature_configuration, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    feature_path = JSE_CACHE_DIR / f"{split_name}_features_{feature_hash}.csv"

    if feature_path.exists():
        frame = pd.read_csv(feature_path, keep_default_na=False)
        expected_rows = (
            len(examples) * len(clustering_configurations)
            * len(weighting_configurations)
        )
        required_columns = {
            "split", "dataset_index", "clustering", "rougeL", "reference",
            "prediction", "answer_type", "predicted_question_type", "vase",
            "semantic_entropy", "joint_se", "cluster_count", "snne",
            "qa_snne_embedding", "length_alpha", "weight_temperature",
        }
        if required_columns - set(frame.columns):
            raise ValueError("Cached PathVQA features are incomplete.")
        if len(frame) != expected_rows:
            raise ValueError("Cached PathVQA feature row count is incorrect.")
        if set(frame["clustering"]) != set(clustering_configurations):
            raise ValueError("Cached PathVQA clustering set is incorrect.")
        print(f"Loaded {split_name} semantic features from {feature_path}")
        return frame, feature_path

    feature_rows = []
    for example in tqdm(examples, desc=f"Semantic clustering: {split_name}"):
        answers = [
            record["answer"] for record in example["records"]
        ]
        qa_sample_example = qa_snne_examples_by_split[split_name][
            int(example["dataset_index"])
        ]
        qa_uncertainty = qa_snne_signals(example, qa_sample_example)
        configurations = jse_clusters_for_configurations(
            answers,
            clustering_configurations,
        )
        rouge_l = jse_rouge_l(example["reference"], example["greedy"])
        answer_type = (
            "Closed (yes/no)"
            if jse_normalize(example["reference"]) in {"yes", "no"}
            else "Open-ended"
        )
        for clustering, cluster_ids in configurations:
            for length_alpha, weight_temperature in weighting_configurations:
                row = {
                    "split": split_name,
                    "dataset_index": int(example["dataset_index"]),
                    "clustering": clustering,
                    "length_alpha": float(length_alpha),
                    "weight_temperature": float(weight_temperature),
                    "rougeL": rouge_l,
                    "reference": example["reference"],
                    "prediction": example["greedy"],
                    "answer_type": answer_type,
                    "predicted_question_type": jse_predict_question_type(
                        example["question"]
                    ),
                }
                row.update(jse_signals(
                    example, cluster_ids, length_alpha, weight_temperature
                ))
                row.update(qa_uncertainty)
                feature_rows.append(row)

    frame = pd.DataFrame(feature_rows)
    frame.to_csv(feature_path, index=False)
    print(f"Saved {split_name} semantic features to {feature_path}")
    return frame, feature_path


jse_validation_features, jse_validation_feature_cache_path = (
    jse_build_feature_frame(
        "validation",
        jse_validation_examples,
        jse_validation_sample_cache_path,
        JSE_SELECTION_CANDIDATES,
        JSE_WEIGHTING_CANDIDATES,
    )
)
display(jse_validation_features.head())
print({
    "dataset": "PathVQA",
    "selection_split": "validation",
    "selection_candidates": JSE_SELECTION_CANDIDATES,
    "length_alpha_grid": LENGTH_ALPHA_GRID,
    "weight_temperature_grid": WEIGHT_TEMPERATURE_GRID,
    "test_features_built_before_selection": False,
})


## 7. Validation selection and locked test safety

Create two validation-only locks with PA-SHE AUROC at `ROUGE-L < 0.50`, using
AUPRC and then candidate order as tie-breakers: one lock from all validation
questions for overall/closed reporting, and one lock from open-ended validation
questions for open-ended reporting. Only after both locks are fixed are official
test features built. Test thresholds `0.30` and `0.70` are sensitivity only.


In [ ]:
JSE_RISK_COLUMNS = {
    "VASE": "vase",
    "SE": "semantic_entropy",
    "SNNE": "snne",
    "QA-SNNE · Embedding": "qa_snne_embedding",
    "PA-SHE": "joint_se",
}


def jse_safe_metrics(labels, scores):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=np.float64)
    if labels.shape != scores.shape:
        raise ValueError("Labels and uncertainty scores must align.")
    if not np.isfinite(scores).all():
        raise ValueError("Uncertainty scores contain NaN or infinity.")
    if np.unique(labels).size < 2:
        return np.nan, np.nan
    return roc_auc_score(labels, scores), average_precision_score(labels, scores)


# Select the closed-question route using question text only.
closed_validation_features = jse_validation_features[
    jse_validation_features["predicted_question_type"] == "Closed"
].copy()
closed_example_count = closed_validation_features["dataset_index"].nunique()
if closed_example_count == 0:
    raise ValueError("Question router predicted no closed validation questions.")
selection_rows = []
for clustering_order, clustering in enumerate(JSE_SELECTION_CANDIDATES):
    for weighting_order, (length_alpha, weight_temperature) in enumerate(
        JSE_WEIGHTING_CANDIDATES
    ):
        candidate_order = (
            clustering_order * len(JSE_WEIGHTING_CANDIDATES) + weighting_order
        )
        group = closed_validation_features[
            (closed_validation_features["clustering"] == clustering)
            & np.isclose(closed_validation_features["length_alpha"], length_alpha)
            & np.isclose(
                closed_validation_features["weight_temperature"],
                weight_temperature,
            )
        ].sort_values("dataset_index")
        if len(group) != closed_example_count:
            raise ValueError(
                f"Validation features are incomplete for {clustering}, "
                f"alpha={length_alpha}, T={weight_temperature}."
            )
        failures = (
            group["rougeL"].to_numpy() < JSE_PRIMARY_LABEL_THRESHOLD
        ).astype(int)
        auroc, auprc = jse_safe_metrics(failures, group["joint_se"])
        selection_rows.append({
            "selection_split": "validation",
            "selection_subset": "Predicted closed",
            "test_used_for_selection": False,
            "candidate_order": candidate_order,
            "label_threshold": JSE_PRIMARY_LABEL_THRESHOLD,
            "clustering": clustering,
            "length_alpha": float(length_alpha),
            "weight_temperature": float(weight_temperature),
            "examples": len(group),
            "failure_prevalence": failures.mean(),
            "validation_AUROC": auroc,
            "validation_AUPRC": auprc,
        })

jse_validation_selection = pd.DataFrame(selection_rows)
ranked_selection = jse_validation_selection.sort_values(
    ["validation_AUROC", "validation_AUPRC", "candidate_order"],
    ascending=[False, False, True],
    kind="mergesort",
)
if ranked_selection.empty or pd.isna(ranked_selection.iloc[0]["validation_AUROC"]):
    raise ValueError("Validation labels cannot select a clustering configuration.")
JSE_LOCKED_CLUSTERING = str(ranked_selection.iloc[0]["clustering"])
JSE_LOCKED_LENGTH_ALPHA = float(ranked_selection.iloc[0]["length_alpha"])
JSE_LOCKED_WEIGHT_TEMPERATURE = float(
    ranked_selection.iloc[0]["weight_temperature"]
)

# Select the open-question route using question text only.
open_validation_features = jse_validation_features[
    jse_validation_features["predicted_question_type"] == "Open"
].copy()
if open_validation_features.empty:
    raise ValueError("PathVQA validation contains no open-ended examples.")

open_selection_rows = []
open_example_count = open_validation_features["dataset_index"].nunique()
for clustering_order, clustering in enumerate(JSE_SELECTION_CANDIDATES):
    for weighting_order, (length_alpha, weight_temperature) in enumerate(
        JSE_WEIGHTING_CANDIDATES
    ):
        candidate_order = (
            clustering_order * len(JSE_WEIGHTING_CANDIDATES) + weighting_order
        )
        group = open_validation_features[
            (open_validation_features["clustering"] == clustering)
            & np.isclose(open_validation_features["length_alpha"], length_alpha)
            & np.isclose(
                open_validation_features["weight_temperature"],
                weight_temperature,
            )
        ].sort_values("dataset_index")
        if len(group) != open_example_count:
            raise ValueError(
                f"Open-ended validation features are incomplete for "
                f"{clustering}, alpha={length_alpha}, T={weight_temperature}."
            )
        failures = (
            group["rougeL"].to_numpy() < JSE_PRIMARY_LABEL_THRESHOLD
        ).astype(int)
        auroc, auprc = jse_safe_metrics(failures, group["joint_se"])
        open_selection_rows.append({
            "selection_split": "validation",
            "selection_subset": "Predicted open",
            "test_used_for_selection": False,
            "candidate_order": candidate_order,
            "label_threshold": JSE_PRIMARY_LABEL_THRESHOLD,
            "clustering": clustering,
            "length_alpha": float(length_alpha),
            "weight_temperature": float(weight_temperature),
            "examples": len(group),
            "failure_prevalence": failures.mean(),
            "validation_AUROC": auroc,
            "validation_AUPRC": auprc,
        })

jse_open_validation_selection = pd.DataFrame(open_selection_rows)
ranked_open_selection = jse_open_validation_selection.sort_values(
    ["validation_AUROC", "validation_AUPRC", "candidate_order"],
    ascending=[False, False, True],
    kind="mergesort",
)
if (
    ranked_open_selection.empty
    or pd.isna(ranked_open_selection.iloc[0]["validation_AUROC"])
):
    raise ValueError(
        "Open-ended validation labels cannot select clustering."
    )
JSE_OPEN_LOCKED_CLUSTERING = str(
    ranked_open_selection.iloc[0]["clustering"]
)
JSE_OPEN_LOCKED_LENGTH_ALPHA = float(
    ranked_open_selection.iloc[0]["length_alpha"]
)
JSE_OPEN_LOCKED_WEIGHT_TEMPERATURE = float(
    ranked_open_selection.iloc[0]["weight_temperature"]
)

# Select the QA-SNNE alignment variant independently on validation only.
qa_validation_base = jse_validation_features[
    (jse_validation_features["clustering"] == JSE_SELECTION_CANDIDATES[0])
    & np.isclose(
        jse_validation_features["length_alpha"], LENGTH_ALPHA_GRID[0]
    )
    & np.isclose(
        jse_validation_features["weight_temperature"],
        WEIGHT_TEMPERATURE_GRID[0],
    )
].sort_values("dataset_index")
qa_failures = (
    qa_validation_base["rougeL"].to_numpy() < JSE_PRIMARY_LABEL_THRESHOLD
).astype(int)
qa_selection_rows = []
for variant_order, (variant, column) in enumerate(QA_SNNE_VARIANTS.items()):
    auroc, auprc = jse_safe_metrics(qa_failures, qa_validation_base[column])
    qa_selection_rows.append({
        "selection_split": "validation",
        "selection_subset": "All",
        "test_used_for_selection": False,
        "variant_order": variant_order,
        "label_threshold": JSE_PRIMARY_LABEL_THRESHOLD,
        "variant": variant,
        "column": column,
        "examples": len(qa_validation_base),
        "failure_prevalence": qa_failures.mean(),
        "validation_AUROC": auroc,
        "validation_AUPRC": auprc,
    })
qa_snne_validation_selection = pd.DataFrame(qa_selection_rows)
qa_ranked = qa_snne_validation_selection.sort_values(
    ["validation_AUROC", "validation_AUPRC", "variant_order"],
    ascending=[False, False, True],
    kind="mergesort",
)
if qa_ranked.empty or pd.isna(qa_ranked.iloc[0]["validation_AUROC"]):
    raise ValueError("Validation labels cannot select a QA-SNNE variant.")
QA_SNNE_LOCKED_VARIANT = str(qa_ranked.iloc[0]["variant"])
QA_SNNE_LOCKED_COLUMN = str(qa_ranked.iloc[0]["column"])
QA_SNNE_LOCKED_METHOD = f"QA-SNNE · {QA_SNNE_LOCKED_VARIANT}"

selection_path = JSE_CACHE_DIR / "validation_selected_clustering.json"
selection_payload = {
    "dataset": "PathVQA",
    "selection_split": "validation",
    "test_used_for_selection": False,
    "question_router_version": JSE_QUESTION_ROUTER_VERSION,
    "routing_input": "question text only; reference answer excluded",
    "primary_label_definition": "ROUGE-L < 0.50",
    "selection_metric": "AUROC; AUPRC tie-breaker; candidate order final tie-breaker",
    "eligible_candidates": JSE_SELECTION_CANDIDATES,
    "length_alpha_grid": LENGTH_ALPHA_GRID,
    "weight_temperature_grid": WEIGHT_TEMPERATURE_GRID,
    "locked_clustering": JSE_LOCKED_CLUSTERING,
    "overall_locked_clustering": JSE_LOCKED_CLUSTERING,
    "closed_route_locked_clustering": JSE_LOCKED_CLUSTERING,
    "overall_locked_length_alpha": JSE_LOCKED_LENGTH_ALPHA,
    "overall_locked_weight_temperature": JSE_LOCKED_WEIGHT_TEMPERATURE,
    "open_ended_locked_clustering": JSE_OPEN_LOCKED_CLUSTERING,
    "open_route_locked_clustering": JSE_OPEN_LOCKED_CLUSTERING,
    "open_ended_locked_length_alpha": JSE_OPEN_LOCKED_LENGTH_ALPHA,
    "open_ended_locked_weight_temperature": (
        JSE_OPEN_LOCKED_WEIGHT_TEMPERATURE
    ),
    "qa_snne_locked_variant": QA_SNNE_LOCKED_VARIANT,
    "qa_snne_locked_column": QA_SNNE_LOCKED_COLUMN,
    "qa_snne_selection_metric": (
        "AUROC; AUPRC tie-breaker; variant order final tie-breaker"
    ),
    "qa_snne_candidates": qa_snne_validation_selection.drop(
        columns="variant_order"
    ).to_dict(orient="records"),
    "overall_candidates": jse_validation_selection.drop(
        columns="candidate_order"
    ).to_dict(orient="records"),
    "open_ended_candidates": jse_open_validation_selection.drop(
        columns="candidate_order"
    ).to_dict(orient="records"),
}
selection_path.write_text(
    json.dumps(selection_payload, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
jse_validation_selection.to_csv(
    JSE_CACHE_DIR / "validation_clustering_selection.csv",
    index=False,
)
jse_open_validation_selection.to_csv(
    JSE_CACHE_DIR / "validation_open_ended_clustering_selection.csv",
    index=False,
)
qa_snne_validation_selection.to_csv(
    JSE_CACHE_DIR / "validation_qa_snne_selection.csv",
    index=False,
)

# Only after both locks: sample test once and build only required features.
qa_snne_test_examples, qa_snne_test_sample_cache_path = (
    qa_snne_collect_split("test", jse_datasets["test"])
)
qa_snne_examples_by_split["test"] = qa_snne_test_examples
qa_snne_sample_cache_paths["test"] = qa_snne_test_sample_cache_path

JSE_TEST_LOCKED_CLUSTERINGS = list(dict.fromkeys([
    JSE_LOCKED_CLUSTERING,
    JSE_OPEN_LOCKED_CLUSTERING,
]))
JSE_TEST_LOCKED_WEIGHTINGS = list(dict.fromkeys([
    (JSE_LOCKED_LENGTH_ALPHA, JSE_LOCKED_WEIGHT_TEMPERATURE),
    (
        JSE_OPEN_LOCKED_LENGTH_ALPHA,
        JSE_OPEN_LOCKED_WEIGHT_TEMPERATURE,
    ),
]))
jse_test_examples, jse_test_sample_cache_path = jse_collect_split(
    "test",
    jse_datasets["test"],
)
jse_examples_by_split["test"] = jse_test_examples
jse_sample_cache_paths["test"] = jse_test_sample_cache_path
jse_test_features, jse_test_feature_cache_path = jse_build_feature_frame(
    "test",
    jse_test_examples,
    jse_test_sample_cache_path,
    JSE_TEST_LOCKED_CLUSTERINGS,
    JSE_TEST_LOCKED_WEIGHTINGS,
)
jse_locked_test_features = jse_test_features[
    (jse_test_features["clustering"] == JSE_LOCKED_CLUSTERING)
    & np.isclose(jse_test_features["length_alpha"], JSE_LOCKED_LENGTH_ALPHA)
    & np.isclose(
        jse_test_features["weight_temperature"],
        JSE_LOCKED_WEIGHT_TEMPERATURE,
    )
    & (jse_test_features["predicted_question_type"] == "Closed")
].sort_values(
    "dataset_index"
).copy()
if jse_locked_test_features["dataset_index"].duplicated().any():
    raise ValueError("Locked PathVQA test features contain duplicate examples.")
jse_open_locked_test_features = jse_test_features[
    (jse_test_features["clustering"] == JSE_OPEN_LOCKED_CLUSTERING)
    & np.isclose(
        jse_test_features["length_alpha"], JSE_OPEN_LOCKED_LENGTH_ALPHA
    )
    & np.isclose(
        jse_test_features["weight_temperature"],
        JSE_OPEN_LOCKED_WEIGHT_TEMPERATURE,
    )
    & (jse_test_features["predicted_question_type"] == "Open")
].sort_values("dataset_index").copy()
if jse_open_locked_test_features["dataset_index"].duplicated().any():
    raise ValueError(
        "Open-ended locked PathVQA test features contain duplicate examples."
    )
if jse_open_locked_test_features.empty:
    raise ValueError("Question router predicted no open test questions.")
jse_routed_test_features = pd.concat(
    [jse_locked_test_features, jse_open_locked_test_features],
    ignore_index=True,
).sort_values("dataset_index")
if (
    len(jse_routed_test_features)
    != jse_test_features["dataset_index"].nunique()
    or jse_routed_test_features["dataset_index"].duplicated().any()
):
    raise ValueError("Question-type routing must cover each test example once.")

closed_validation_locked = closed_validation_features[
    (closed_validation_features["clustering"] == JSE_LOCKED_CLUSTERING)
    & np.isclose(closed_validation_features["length_alpha"], JSE_LOCKED_LENGTH_ALPHA)
    & np.isclose(
        closed_validation_features["weight_temperature"],
        JSE_LOCKED_WEIGHT_TEMPERATURE,
    )
].sort_values("dataset_index")
open_validation_locked = open_validation_features[
    (open_validation_features["clustering"] == JSE_OPEN_LOCKED_CLUSTERING)
    & np.isclose(
        open_validation_features["length_alpha"], JSE_OPEN_LOCKED_LENGTH_ALPHA
    )
    & np.isclose(
        open_validation_features["weight_temperature"],
        JSE_OPEN_LOCKED_WEIGHT_TEMPERATURE,
    )
].sort_values("dataset_index")

def jse_validation_percentile(validation_scores, query_scores):
    reference = np.sort(np.asarray(validation_scores, dtype=float))
    query = np.asarray(query_scores, dtype=float)
    if reference.size == 0:
        raise ValueError("Cannot calibrate from an empty validation subset.")
    return np.searchsorted(reference, query, side="right") / reference.size

for _, risk_column in JSE_RISK_COLUMNS.items():
    calibrated_column = f"calibrated_{risk_column}"
    jse_locked_test_features[calibrated_column] = jse_validation_percentile(
        closed_validation_locked[risk_column],
        jse_locked_test_features[risk_column],
    )
    jse_open_locked_test_features[calibrated_column] = jse_validation_percentile(
        open_validation_locked[risk_column],
        jse_open_locked_test_features[risk_column],
    )
jse_routed_test_features = pd.concat(
    [jse_locked_test_features, jse_open_locked_test_features],
    ignore_index=True,
).sort_values("dataset_index")
jse_features = jse_routed_test_features

evaluation_subsets = {
    "All": jse_routed_test_features,
    "Closed (yes/no)": jse_routed_test_features[
        jse_routed_test_features["answer_type"] == "Closed (yes/no)"
    ],
    "Open-ended": jse_routed_test_features[
        jse_routed_test_features["answer_type"] == "Open-ended"
    ],
}

jse_result_rows = []
for evaluation_subset, subset_features in evaluation_subsets.items():
    subset_locked_clustering = "Question-type routed"
    for label_threshold in JSE_LABEL_THRESHOLDS:
        failures = (
            subset_features["rougeL"].to_numpy() < label_threshold
        ).astype(int)
        for method, column in JSE_RISK_COLUMNS.items():
            score_column = f"calibrated_{column}"
            auroc, auprc = jse_safe_metrics(failures, subset_features[score_column])
            jse_result_rows.append({
                "evaluation_split": "official test",
                "evaluation_subset": evaluation_subset,
                "examples": len(subset_features),
                "label_threshold": label_threshold,
                "failure_prevalence": failures.mean() if len(failures) else np.nan,
                "clustering": subset_locked_clustering,
                "length_alpha": np.nan,
                "weight_temperature": np.nan,
                "method": method,
                "AUROC": auroc,
                "AUPRC": auprc,
            })

jse_results = pd.DataFrame(jse_result_rows)
jse_primary_results = jse_results[
    (jse_results["evaluation_subset"] == "All")
    & np.isclose(jse_results["label_threshold"], JSE_PRIMARY_LABEL_THRESHOLD)
].sort_values(["AUROC", "AUPRC"], ascending=False)
jse_label_sensitivity = jse_results[
    (jse_results["evaluation_subset"] == "All")
    & (jse_results["method"] == "PA-SHE")
].sort_values("label_threshold")
jse_open_ended_results = jse_results[
    jse_results["evaluation_subset"] == "Open-ended"
].copy()
jse_open_ended_primary_results = jse_open_ended_results[
    np.isclose(
        jse_open_ended_results["label_threshold"],
        JSE_PRIMARY_LABEL_THRESHOLD,
    )
].sort_values(["AUROC", "AUPRC"], ascending=False)
jse_open_ended_label_sensitivity = jse_open_ended_results[
    jse_open_ended_results["method"] == "PA-SHE"
].sort_values("label_threshold")

jse_results.to_csv(JSE_CACHE_DIR / "locked_test_safety_results.csv", index=False)
jse_open_ended_results.to_csv(
    JSE_CACHE_DIR / "locked_test_open_ended_safety_results.csv",
    index=False,
)

print("VALIDATION-ONLY QA-SNNE variant selection:")
display(qa_snne_validation_selection.drop(columns="variant_order").round(4))
print("Locked QA-SNNE variant before test sampling:", QA_SNNE_LOCKED_VARIANT)
print("VALIDATION-ONLY PathVQA clustering selection:")
display(jse_validation_selection.drop(columns="candidate_order").round(4))
print("VALIDATION-ONLY open-ended clustering selection:")
display(
    jse_open_validation_selection.drop(columns="candidate_order").round(4)
)
print("Closed-route lock before test sampling:", JSE_LOCKED_CLUSTERING)
print("Closed-route locked weighting:", {
    "length_alpha": JSE_LOCKED_LENGTH_ALPHA,
    "weight_temperature": JSE_LOCKED_WEIGHT_TEMPERATURE,
})
print(
    "Open-route lock before test sampling:",
    JSE_OPEN_LOCKED_CLUSTERING,
)
print("Open-route locked weighting:", {
    "length_alpha": JSE_OPEN_LOCKED_LENGTH_ALPHA,
    "weight_temperature": JSE_OPEN_LOCKED_WEIGHT_TEMPERATURE,
})
print("OFFICIAL TEST primary results: failure = ROUGE-L < 0.50")
display(jse_primary_results.round(4))
print("OFFICIAL TEST open-ended primary results")
display(jse_open_ended_primary_results.round(4))
print("PA-SHE test label sensitivity; clustering remains locked")
display(jse_label_sensitivity.round(4))
print("Open-ended PA-SHE label sensitivity; clustering remains locked")
display(jse_open_ended_label_sensitivity.round(4))
print("Saved validation selection protocol:", selection_path)


## 8. Validation diagnostic and locked test comparisons

The first plot compares the overall and open-ended validation-only clustering
selections. Official-test overall results use the overall lock; open-ended
official-test results use the independently selected open-ended lock.


In [ ]:
# Validation-only diagnostics: overall and open-ended selection paths.
validation_plots = [
    ("Predicted-closed validation", jse_validation_selection),
    ("Predicted-open validation", jse_open_validation_selection),
]
fig, axes = plt.subplots(2, 2, figsize=(20, 10))
for row_index, (subset_label, selection_frame) in enumerate(validation_plots):
    validation_plot = selection_frame.sort_values("candidate_order")
    axes[row_index, 0].bar(
        validation_plot["clustering"],
        validation_plot["validation_AUROC"],
        color="#4c78a8",
    )
    axes[row_index, 1].bar(
        validation_plot["clustering"],
        validation_plot["validation_AUPRC"],
        color="#f58518",
    )
    axes[row_index, 0].set_title(f"{subset_label}: PA-SHE AUROC")
    axes[row_index, 1].set_title(f"{subset_label}: PA-SHE AUPRC")
    for axis in axes[row_index]:
        axis.set_ylim(0, 1)
        axis.tick_params(axis="x", rotation=45)
        axis.grid(axis="y", alpha=0.25)
fig.suptitle("PathVQA validation locks: ROUGE-L < 0.50")
plt.tight_layout()
plt.show()

# Official-test label sensitivity; clustering remains locked.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(
    jse_label_sensitivity["label_threshold"],
    jse_label_sensitivity["AUROC"],
    marker="o",
)
axes[1].plot(
    jse_label_sensitivity["label_threshold"],
    jse_label_sensitivity["AUPRC"],
    marker="o",
    color="#f58518",
)
axes[0].set_title("Official test AUROC sensitivity")
axes[1].set_title("Official test AUPRC sensitivity")
for axis in axes:
    axis.set_xlabel("ROUGE-L failure-label threshold")
    axis.set_ylim(0, 1)
    axis.set_xticks(JSE_LABEL_THRESHOLDS)
    axis.grid(alpha=0.25)
fig.suptitle("PA-SHE with validation-locked question-type routing")
plt.tight_layout()
plt.show()

primary_comparison = jse_primary_results.sort_values("AUROC")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(primary_comparison["method"], primary_comparison["AUROC"])
axes[1].barh(
    primary_comparison["method"],
    primary_comparison["AUPRC"],
    color="#d17a22",
)
axes[0].set_title("Official test AUROC")
axes[1].set_title("Official test AUPRC")
for axis in axes:
    axis.set_xlim(0, 1)
    axis.grid(axis="x", alpha=0.25)
fig.suptitle(
    "Validation-locked question-type routing; "
    f"failure = ROUGE-L < {JSE_PRIMARY_LABEL_THRESHOLD:.2f}"
)
plt.tight_layout()
plt.show()


## 9. Locked selective prediction and audit

These overall analyses use official-test features from the overall
validation-locked clustering configuration.


In [ ]:
primary_features = jse_routed_test_features.copy()
rejection_fractions = np.linspace(0, 0.50, 11)
curve_rows = []

for method, column in JSE_RISK_COLUMNS.items():
    ordered = primary_features.sort_values(
        f"calibrated_{column}", ascending=True
    )
    for fraction in rejection_fractions:
        retained_count = max(1, int(round(len(ordered) * (1.0 - fraction))))
        retained = ordered.iloc[:retained_count]
        curve_rows.append({
            "method": method,
            "rejected_fraction": fraction,
            "retained_ROUGE-L": retained["rougeL"].mean(),
        })

jse_rejection = pd.DataFrame(curve_rows)
plt.figure(figsize=(11, 6))
for method, group in jse_rejection.groupby("method", sort=False):
    plt.plot(
        group["rejected_fraction"],
        group["retained_ROUGE-L"],
        marker="o",
        label=method,
    )
plt.xlabel("Fraction rejected as high risk")
plt.ylabel("Mean ROUGE-L among retained answers")
plt.title("Official test selective prediction: question-type routed")
plt.grid(alpha=0.25)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

audit_columns = [
    "dataset_index", "answer_type", "reference", "prediction", "rougeL",
    "joint_se", "semantic_entropy",
]
print("Highest PA-SHE official-test cases")
display(primary_features.nlargest(10, "joint_se")[audit_columns].round(4))
print("Low PA-SHE official-test failures")
display(
    primary_features[
        primary_features["rougeL"] < JSE_PRIMARY_LABEL_THRESHOLD
    ].nsmallest(10, "joint_se")[audit_columns].round(4)
)


## 10. Reporting checklist

- Report the 13 PathVQA candidates: Exact plus three thresholds for each of SBERT, BGE, RoBERTa-NLI, and DeBERTa-NLI.
- Report overall selection on all official validation examples at `ROUGE-L < 0.50`.
- Report open-ended selection on open-ended official validation examples only.
- Report validation AUROC selection, AUPRC tie-break, then candidate-order tie-break.
- Lock both configurations before building any official-test features.
- Use the overall lock for all/closed results and the open-ended lock only for open-ended results.
- Treat test labels `0.30` and `0.70` as sensitivity analyses, not selection.
- Report overall, closed-ended, and open-ended test AUROC/AUPRC separately.


## 11. Main comparison table

The table uses an overall clustering selected from all validation questions and
an open-ended clustering selected from open-ended validation questions. Both are
locked before test feature construction. Utility is shared because all risk
methods evaluate the same deployed greedy predictions. Closed-ended safety
uses exact normalized `yes`/`no` references; all other references are open-ended.


In [ ]:
# Final comparison using the two validation-locked evaluation paths.
import evaluate
from IPython.display import display

comparison_required = [
    "jse_results", "jse_open_ended_results", "jse_routed_test_features",
    "JSE_LOCKED_CLUSTERING", "JSE_OPEN_LOCKED_CLUSTERING",
    "JSE_PRIMARY_LABEL_THRESHOLD",
    "QA_SNNE_LOCKED_VARIANT", "QA_SNNE_LOCKED_METHOD",
]
comparison_missing = [
    name for name in comparison_required if name not in globals()
]
if comparison_missing:
    raise RuntimeError(
        "Run PA-SHE sections 5-7 first. Missing: "
        + ", ".join(comparison_missing)
    )

utility_source = jse_routed_test_features.sort_values(
    "dataset_index"
).drop_duplicates("dataset_index")
references = [
    normalize_vqa_metric_text(text)
    for text in utility_source["reference"].astype(str)
]
predictions = [
    normalize_vqa_metric_text(text)
    for text in utility_source["prediction"].astype(str)
]
comparison_utility = {
    "BLEU": 100.0 * float(evaluate.load("bleu").compute(
        predictions=predictions, references=references
    )["bleu"]),
    "ROUGE-L": 100.0 * float(evaluate.load("rouge").compute(
        predictions=predictions, references=references
    )["rougeL"]),
    "METEOR": 100.0 * float(evaluate.load("meteor").compute(
        predictions=predictions, references=references
    )["meteor"]),
}

overall_safety = jse_results[
    (jse_results["evaluation_subset"] == "All")
    & np.isclose(jse_results["label_threshold"], JSE_PRIMARY_LABEL_THRESHOLD)
].copy()
closed_safety = jse_results[
    (jse_results["evaluation_subset"] == "Closed (yes/no)")
    & np.isclose(jse_results["label_threshold"], JSE_PRIMARY_LABEL_THRESHOLD)
].copy()
open_safety = jse_open_ended_results[
    np.isclose(
        jse_open_ended_results["label_threshold"],
        JSE_PRIMARY_LABEL_THRESHOLD,
    )
].copy()


def comparison_safety(frame, method):
    match = frame[frame["method"] == method]
    if len(match) != 1:
        raise ValueError(f"Expected one locked result for {method}; found {len(match)}")
    row = match.iloc[0]
    return 100.0 * float(row["AUROC"]), 100.0 * float(row["AUPRC"])


locked_variant = (
    f"Closed route={JSE_LOCKED_CLUSTERING}, alpha={JSE_LOCKED_LENGTH_ALPHA:g}, "
    f"T={JSE_LOCKED_WEIGHT_TEMPERATURE:g}; "
    f"Open route={JSE_OPEN_LOCKED_CLUSTERING}, "
    f"alpha={JSE_OPEN_LOCKED_LENGTH_ALPHA:g}, "
    f"T={JSE_OPEN_LOCKED_WEIGHT_TEMPERATURE:g}"
)
specification = [
    ("Semantic entropy", locked_variant, "SE"),
    ("Semantic nearest-neighbour entropy", f"ROUGE-L · n={QA_SNNE_NUM_SAMPLES}", "SNNE"),
    ("Visual stability", locked_variant, "VASE"),
    ("Perturbation-Aware Semantic Hallucination Entropy (PA-SHE)", locked_variant, "PA-SHE"),
]
for qa_variant in QA_SNNE_VARIANTS:
    selected_suffix = (
        " · validation-selected"
        if qa_variant == QA_SNNE_LOCKED_VARIANT else ""
    )
    specification.append((
        "Question-aligned SNNE",
        f"{qa_variant}{selected_suffix} · beta={QA_SNNE_BETA:g}",
        f"QA-SNNE · {qa_variant}",
    ))

rows = []
for family, variant, method in specification:
    overall_auroc, overall_auprc = comparison_safety(overall_safety, method)
    closed_auroc, closed_auprc = comparison_safety(closed_safety, method)
    open_auroc, open_auprc = comparison_safety(open_safety, method)
    rows.append({
        "Uncertainty method": family,
        "Variant / clustering": variant,
        ("Utility", "BLEU"): comparison_utility["BLEU"],
        ("Utility", "ROUGE-L"): comparison_utility["ROUGE-L"],
        ("Utility", "METEOR"): comparison_utility["METEOR"],
        ("Overall safety", "AUROC"): overall_auroc,
        ("Overall safety", "AUPRC"): overall_auprc,
        ("Closed-ended safety", "AUROC"): closed_auroc,
        ("Closed-ended safety", "AUPRC"): closed_auprc,
        ("Open-ended safety", "AUROC"): open_auroc,
        ("Open-ended safety", "AUPRC"): open_auprc,
    })

comparison_df = pd.DataFrame(rows).set_index([
    "Uncertainty method", "Variant / clustering"
])
comparison_df.columns = pd.MultiIndex.from_tuples(
    comparison_df.columns,
    names=["Evaluation dimension", "Metric"],
)
safety_columns = [
    ("Overall safety", "AUROC"), ("Overall safety", "AUPRC"),
    ("Closed-ended safety", "AUROC"), ("Closed-ended safety", "AUPRC"),
    ("Open-ended safety", "AUROC"), ("Open-ended safety", "AUPRC"),
]
safety_maxima = {column: comparison_df[column].max() for column in safety_columns}


def highlight_maximum(value, column):
    maximum = safety_maxima.get(column)
    if maximum is not None and pd.notna(value) and np.isclose(value, maximum):
        return "font-weight: 700; background-color: #e8f1fb;"
    return ""


comparison_styler = (
    comparison_df.style
    .format("{:.2f}", na_rep="—")
    .apply(
        lambda series: [highlight_maximum(value, series.name) for value in series],
        axis=0,
    )
    .set_caption(
        "PathVQA: Overall, Closed-Ended, and Open-Ended Safety "
        "with Validation-Locked Routing"
    )
    .set_table_styles([
        {"selector": "caption", "props": [
            ("caption-side", "top"), ("font-size", "18px"),
            ("font-weight", "700"), ("text-align", "left"),
        ]},
        {"selector": "th", "props": [
            ("background-color", "#f5f5f5"), ("border", "1px solid #aaa"),
            ("padding", "8px"), ("text-align", "center"),
        ]},
        {"selector": "td", "props": [
            ("border", "1px solid #b5b5b5"), ("padding", "8px"),
            ("text-align", "center"),
        ]},
        {"selector": "table", "props": [
            ("border-collapse", "collapse"), ("font-size", "13px"),
            ("width", "100%"),
        ]},
    ])
)
display(comparison_styler)
comparison_df.to_csv("PathVQA_Joint_SE_main_comparison_table.csv")
with open(
    "PathVQA_Joint_SE_main_comparison_table.html",
    "w",
    encoding="utf-8",
) as comparison_file:
    comparison_file.write(comparison_styler.to_html())
print("Closed-route locked clustering:", JSE_LOCKED_CLUSTERING)
print("Open-route locked clustering:", JSE_OPEN_LOCKED_CLUSTERING)
print("Saved PathVQA_Joint_SE_main_comparison_table.csv/html")


### Reading the table

- Utility scores describe the frozen VQA answer model and repeat across risks.
- Overall safety uses all test questions; closed-ended safety uses exact normalized `yes`/`no` references; open-ended safety uses all remaining references.
- AUROC and AUPRC detect failures defined by the primary `ROUGE-L < 0.50` label.
- Semantic rows show the overall lock and the separately selected open-ended lock.
- Both locks are selected without official-test labels or scores.
- Bold cells indicate the best displayed safety ranking, not a new test selection.
